# Проект "Предсказание покупок в интернет-магазине"

## Описание проекта

Интернет-магазин собирает историю покупателей, проводит рассылки предложений и планирует будущие продажи. 

Для оптимизации процессов надо выделить пользователей, которые готовы совершить покупку в ближайшее время.

### Цель

Предсказать вероятность покупки в течение 90 дней.

### Задачи

- Изучить данные
- Разработать полезные признаки
- Создать модель для классификации пользователей
- Улучшить модель и максимизировать метрику roc_auc
- Выполнить тестирование

## Описание данных

Данные о покупках клиентов по дням и по товарам. 

### Таблица о товаре, его цена, количество штук (apparel-purchases).

В таблице есть списки идентификаторов, к каким категориям относится товар. Часто это вложенные категории (например автотовары-аксессуары-освежители), но также может включать в начале списка маркер распродажи или маркер женщинам/мужчинам.

- `client_id` — идентификатор клиента,
- `quantity` — количество единиц товара,
- `price` — цена товара,
- `category_ids` — идентификаторы категорий,
- `date` — дата покупки,
- `message_id` — идентификатор сообщения из рассылки.

### Таблица покупок (apparel-messages).

Рассылки, которые были отправлены клиентам из таблицы покупок.

- `bulk_campaign_id` — идентификатор рассылки,
- `client_id` — идентификатор клиента,
- `message_id` — идентификатор сообщения,
- `event` — действие с сообщением (отправлено, открыто, покупка …),
- `channel` — канал рассылки,
- `date` — дата действия,
- `created_at` — точное время создания сообщения.

### Целевой показатель (target).

- `client_id` — идентификатор клиента,
- `target` — клиент совершил повторную покупку в целевом периоде.

## Результат Репозиторий на гитхабе:

- тетрадь jupyter notebook с описанием, подготовкой признаков, обучением модели и тестированием
- описание проекта и инструкция по использованию в файле README.md
- список зависимостей в файле requirements.txt

## Ссылка на GitHab



https://github.com/Ckomopox80/Project_predicting_online_shopping.git

## Загрузка данных.

Данные о покупках клиентов по дням и по товарам. 

В каждой записи покупка
определенного товара, его цена, количество штук (apparel-purchases).

* client_id - идентификатор клиента
* quantity - количество единиц товара
* price - цена товара
* category_ids - идентификаторы категорий
* date - дата покупки
* message_id - идентификатор сообщения из рассылки

Рассылки, которые были отправлены клиентам из таблицы покупок (apparel-messages).

* bulk_campaign_id - идентификатор рассылки
* client_id - идентификатор клиента
* message_id - идентификатор сообщения
* event - действие с сообщением (отправлено, открыто, покупка…)
* channel - канал рассылки
* date - дата действия
* created_at - дата-время полностью

Целевой показатель (target).

* client_id - идентификатор клиента
* target - клиент совершил покупку в целевом периоде

### Установка библиотек.

In [6]:
# Устанавливаем необходимые библиотеки
!pip install xgboost -q

### Импорт библиотек.

In [7]:
# Импорт функции из стандартной библиотеки для безопасного преобразования строк в объекты Python
from ast import literal_eval

# Импорт широко используемых библиотек для работы с данными и научными вычислениями
import pandas as pd
import numpy as np

# Импорт библиотек для построения графиков и визуализации данных
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, ScalarFormatter  # Форматирование осей графиков
import seaborn as sns  # Расширенная библиотека визуализации на основе matplotlib

# Импорт библиотеки для расширенного вычисления коэффициентов корреляции (Phik)
import phik

# Импорт модулей для разделения данных и подборов параметров модели
from sklearn.model_selection import train_test_split, GridSearchCV

# Импорт инструментов для создания пайплайнов обработки и трансформации данных
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer  # Заполнение пропусков

# Импорт методов предобработки данных
from sklearn.preprocessing import (
    OneHotEncoder,  # Кодирование категориальных признаков
    MinMaxScaler,   # Масштабирование признаков
)

# Импорт классической модели логистической регрессии
from sklearn.linear_model import LogisticRegression

# Импорт популярных градиентных бустинговых моделей
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Импорт простой «фиктивной» модели для базового сравнения
from sklearn.dummy import DummyClassifier

# Импорт метрик для оценки качества классификации
from sklearn.metrics import (
    precision_score,
    recall_score,
    fbeta_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve,
)

# Импорт для оценки важности признаков через перестановки
from sklearn.inspection import permutation_importance

# Импорт библиотеки для интерпретируемости моделей машинного обучения SHAP
import shap


### Настройки.

In [8]:
# Настройка pandas для отображения всех столбцов и строк
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# Импортируем функции display и класс Markdown из IPython.display для форматированного вывода текста в Jupyter Notebook
from IPython.display import display, Markdown

# воспроизводимость результатов
RANDOM_STATE = 42
TEST_SIZE = 0.25

### Функции

In [9]:
# Функция принимает число в десятичном формате и возвращает строку с процентным представлением
# с точностью до двух знаков после запятой (например, 0.1234 -> "12.34%")
def format_percent(value):
    return f"{value:.2%}"

In [10]:
# Функция форматирует число для вывода:
# - Если число целое, выводит его без десятичных знаков, разделяя тысячи пробелами
# - Если число дробное, выводит с двумя знаками после запятой и пробелами для разделения тысяч
# - Если входные данные невозможно преобразовать в число, возвращает исходное значение без изменений
def format_number(value):
    try:
        value = float(value)
        if value.is_integer():
            return f'{value:,.0f}'.replace(',', ' ')
        else:
            return f'{value:,.2f}'.replace(',', ' ')
    except (ValueError, TypeError):
        return value


In [11]:
# Выводит количество записей в столбце field DataFrame data,
# где значения имеют дробную часть (не являются целыми числами)
def print_numeric_with_fraction_count(data, field):
    print(f"Количество записей с дробной частью: {has_fraction(data, field)[field].count()}")


In [12]:
# Функция возвращает строки DataFrame data, в которых указанный столбец field содержит значения с дробной частью,
# при этом пропуски (NaN) исключаются из рассмотрения
def has_fraction(data, field):
    return data[(~data[field].isna()) & (data[field] % 1 != 0)]


In [13]:
# Выводит количество записей в столбце field DataFrame data,
# содержащих значения с дробной частью (т.е. нецелые числа)
def print_numeric_with_fraction_count(data, field):
    print(f"Количество записей с дробной частью: {has_fraction(data, field)[field].count()}")


In [14]:
# Функция генерирует описательные статистики для числовых столбцов DataFrame,
# включая указанные перцентили, переименовывает метрики в удобочитаемые на русском языке,
# и форматирует числовые значения с помощью внешней функции format_number
def describe_numeric(data, percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]):
    # Получаем стандартное статистическое описание данных с указанными перцентилями
    stats = data.describe(percentiles=percentiles)
    
    # Словарь для русификации названий статистик
    columns_desc = {
        'count': 'Количество',
        'mean': 'Среднее',
        'std': 'Станд. отклонение',  # Здесь, возможно, опечатка: "Стандартное отклонение"
        'min': 'Минимум',
        'max': 'Максимум',
        '1%': '1-й процентиль',
        '5%': '5-й процентиль',
        '25%': '25-й процентиль',
        '50%': 'Медиана',
        '75%': '75-й процентиль',
        '95%': '95-й процентиль',
        '99%': '99-й процентиль'
    }
    
    # Переименовываем индексы статистик и форматируем значения
    stats = stats.rename(index=columns_desc).apply(format_number)
    
    # Возвращаем преобразованный DataFrame со статистиками
    return stats


In [15]:
# Функция анализирует пропуски в DataFrame data, учитывая различные варианты обозначения отсутствующих значений,
# и формирует отчёт с количеством и долей пропусков для каждого столбца.
def analyze_na_values(data, fields_descriptions):
    # Список значений, считающихся пропущенными, включая стандартные NaN и различные строковые варианты
    missing_values = [np.nan, None, 'NaN', 'nan', 'NA', 'N/A', 'null', '', ' ']
    
    # Для каждого столбца создаётся маска пропусков:
    # - Если столбец строкового типа (object), проверяем, содержится ли значение в списке missing_values
    # - Для остальных типов используем стандартную функцию isna()
    null_mask = data.apply(lambda col: 
        col.isin(missing_values) if col.dtype == 'object' else col.isna()
    )

    # Общее количество строк в DataFrame
    total_rows = data.shape[0]
    
    # Считаем количество пропусков в каждом столбце
    null_stats = null_mask.sum()
    
    # Оставляем только столбцы с пропусками и сортируем по убыванию количества
    null_stats = null_stats[null_stats > 0].sort_values(ascending=False)

    # Создаём DataFrame с информацией о пропусках:
    # 'Поле' — название столбца,
    # 'Тип данных' — тип данных столбца,
    # 'Количество пропусков' — число пропусков,
    # 'Доля пропусков (%)' — процент пропусков относительно общего числа строк
    report = pd.DataFrame({
        'Поле': null_stats.index,
        'Тип данных': data[null_stats.index].dtypes.values,
        'Количество пропусков': null_stats.values,
        'Доля пропусков (%)': (null_stats / total_rows * 100).round(2)
    })

    # При наличии словаря описаний добавляем колонку с описаниями полей
    if fields_descriptions:
        report['Описание'] = report['Поле'].map(fields_descriptions)

    # Возвращаем итоговый отчёт
    return report


In [16]:
# Функция анализирует уникальные значения в каждом столбце DataFrame,
# формируя статистику по числу уникальных значений, их доле,
# а также по наиболее частому значению и его встречаемости.
def analyze_unique_values(data, fields_descriptions):
    total_rows = data.shape[0]  # Общее число строк в данных
    unique_stats = []  # Список для хранения результатов анализа по каждому столбцу
    
    # Проходим по всем столбцам DataFrame
    for column in data.columns:
        unique_count = data[column].nunique(dropna=True)  # Количество уникальных значений с пропусками исключая NaN
        unique_ratio = unique_count / total_rows  # Доля уникальных значений от общего числа записей

        if unique_count > 0:
            value_count = data[column].value_counts()  # Подсчёт количества каждого уникального значения
            top_value = value_count.index[0]  # Самое частое значение
            top_value_count = value_count.iloc[0]  # Количество вхождений самого частого значения
            top_value_ratio = top_value_count / total_rows  # Доля самого частого значения
        else:
            # Если уникальных значений нет (например, столбец полностью пустой), устанавливаем нулевые показатели
            top_value = None
            top_value_count = 0
            top_value_ratio = 0.0
        
        # Добавляем статистику по текущему столбцу в список
        unique_stats.append({
            'Поле': column,
            'Наименование поля': fields_descriptions.get(column, ''),  # Подставляем описание из словаря, если есть
            'Уникальных значений': unique_count,
            'Доля уникальных (%)': unique_ratio,
            'Самое частое значение': top_value,
            'Кол-во самого частого значения': top_value_count,
            'Доля самого частого значения (%)': top_value_ratio
        })
    
    # Преобразуем список с результатами в DataFrame для наглядности
    df = pd.DataFrame(unique_stats)
    # Сортируем по убыванию доли уникальных значений
    df = df.sort_values(by='Доля уникальных (%)', ascending=False)
    
    # Форматируем доли в проценты без десятичных знаков для удобного восприятия
    df['Доля уникальных (%)'] = df['Доля уникальных (%)'].apply(lambda x: f"{x:.0%}")
    df['Доля самого частого значения (%)'] = df['Доля самого частого значения (%)'].apply(lambda x: f"{x:.0%}")
    
    # Возвращаем итоговый DataFrame с анализом уникальных значений
    return df


In [17]:
# Функция форматирует число x адаптивно в зависимости от его величины:
# - Если число равно нулю, возвращает строку "0"
# - Если абсолютное значение от 0.001 до 1, форматирует с точностью до 4 знаков после запятой,
#   убирая незначащие нули справа и лишнюю десятичную точку
# - Если абсолютное значение меньше 0.001, форматирует с точностью до 6 знаков после запятой
# - Если абсолютное значение больше или равно 1000, форматирует без десятичных знаков
# - В остальных случаях форматирует с 2 знаками после запятой, убирая незначащие нули и точки
def format_adaptive(x, pos=None):
    x = float(x)
    abs_x = abs(x)
    
    if abs_x == 0:
        return "0"
    elif 0.001 <= abs_x < 1:
        return f"{x:.4f}".rstrip('0').rstrip('.') if '.' in f"{x:.4f}" else f"{x:.4f}"
    elif abs_x < 0.001:
        return f"{x:.6f}".rstrip('0').rstrip('.') if '.' in f"{x:.6f}" else f"{x:.6f}"
    elif abs_x >= 1000:
        return f"{x:.0f}"
    else:
        return f"{x:.2f}".rstrip('0').rstrip('.') if '.' in f"{x:.2f}" else f"{x:.2f}"


In [18]:
# Функция строит для указанных числовых полей DataFrame комбинированные графики:
# гистограмму распределения и боксплот, с настройкой внешнего вида и статистических аннотаций.
def draw_combined_hist_boxplot(data, fields_description, fields=None, field_data_types=['number'],
                               figsize=(16, 4), bins=None, showmeans=True, dpi=100):
    # Определяем поля для построения:
    # если fields не указан, берём все столбцы с заданными типами данных (по умолчанию числовые)
    if fields is None:
        fields_to_plot = data.select_dtypes(include=field_data_types).columns
    else:
        # Если указан список полей, проверяем их существование и числовой тип
        fields_to_plot = []
        for field in fields:
            if field not in data.columns:
                raise ValueError(f"Поле '{field}' не найдено в данных")
            if not pd.api.types.is_numeric_dtype(data[field]):
                raise ValueError(f"Поле '{field}' не является числовым")
            fields_to_plot.append(field)

    # Если нет подходящих полей, выводим предупреждение и прекращаем выполнение
    if len(fields_to_plot) == 0:
        print("Нет числовых полей для отображения")
        return
    
    for i, field in enumerate(fields_to_plot):
        clean_data = data[field].dropna()  # Убираем пропущенные значения
        
        if len(clean_data) == 0:
            print(f"Поле '{field}' не содержит данных после удаления NaN")
            continue
        
        # Вычисляем основной статистический размах и количество уникальных значений
        data_range = clean_data.max() - clean_data.min()
        unique_values = len(clean_data.unique())
        
        # Получаем описание поля для заголовка
        description = fields_description.get(field, "нет описания")
        title = (f'{field} - {description}\n'
                 f'n={len(clean_data):,}, mean={clean_data.mean():.2f}, '
                 f'std={clean_data.std():.2f}')
        
        # Создаём фигуру с двумя осями: для гистограммы (левая, большая) и боксплота (правая, узкая)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=figsize, dpi=dpi,
                                       gridspec_kw={'width_ratios': [2, 1]})
        
        # Определяем количество интервалов (бинов) для гистограммы, если не задано явно
        if bins is None:
            bins_sturges = int(1 + np.log2(len(clean_data))) if len(clean_data) > 0 else 10
            bins_auto = min(30, unique_values) if unique_values > 1 else 10
            hist_bins = min(bins_sturges, bins_auto)
        else:
            hist_bins = bins
        
        max_data = clean_data.max()
        min_data = clean_data.min()
        
        # Настраиваем padding и форматирование для оси X гистограммы в зависимости от размаха данных
        if data_range < 0.1:
            padding = max(data_range * 0.1, 0.01)
            xlim_hist = (min_data - padding, max_data + padding)
            formatter_hist = FuncFormatter(lambda x, _: f"{x:.6f}")
        else:
            padding = max(data_range * 0.05, abs(min_data) * 0.05 if min_data != 0 else 0.1)
            xlim_hist = (min_data - padding, max_data + padding)
            formatter_hist = FuncFormatter(format_adaptive)
        
        # Строим гистограмму с заданным количеством бинов
        n, bins_edges, patches = ax1.hist(clean_data, bins=hist_bins, 
                                          edgecolor='black', alpha=0.7,
                                          color='skyblue')
        
        # Рассчитываем и отображаем важные статистики: среднее, медиану и стандартное отклонение
        mean_val = clean_data.mean()
        median_val = clean_data.median()
        std_val = clean_data.std()
        
        ax1.axvline(mean_val, color='red', linestyle='--', linewidth=1.5, 
                   label=f'Среднее: {mean_val:.2f}')
        ax1.axvline(median_val, color='green', linestyle='--', linewidth=1.5,
                   label=f'Медиана: {median_val:.2f}')
        ax1.axvline(mean_val - std_val, color='orange', linestyle=':', linewidth=1,
                   alpha=0.7, label='±1σ')
        ax1.axvline(mean_val + std_val, color='orange', linestyle=':', linewidth=1,
                   alpha=0.7)
        
        # Оформляем оси, сетку и легенду гистограммы
        ax1.set_title('Гистограмма распределения', fontsize=11, fontweight='bold')
        ax1.set_xlabel('Значение', fontsize=10)
        ax1.set_ylabel('Частота', fontsize=10)
        ax1.set_xlim(xlim_hist)
        ax1.xaxis.set_major_formatter(formatter_hist)
        ax1.grid(True, linestyle=':', alpha=0.5)
        ax1.legend(fontsize=9)
        
        # Добавляем текст с основными статистиками на график
        stats_text = (f'Минимум: {min_data:.2f}\n'
                      f'Максимум: {max_data:.2f}\n'
                      f'Размах: {data_range:.2f}\n'
                      f'Уникальных: {unique_values}')
        ax1.text(0.02, 0.98, stats_text,
                 transform=ax1.transAxes, fontsize=9,
                 verticalalignment='top',
                 bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
        
        # Параметры оформления боксплота
        boxprops = dict(linestyle='-', linewidth=1.5, color='darkblue')
        whiskerprops = dict(linestyle='-', linewidth=1.5, color='darkblue')
        medianprops = dict(linestyle='-', linewidth=2, color='red')
        meanprops = dict(marker='D', markersize=6, markerfacecolor='green', markeredgecolor='green')
        
        # Строим вертикальный боксплот с отображением средней метки
        bp = ax2.boxplot(clean_data, vert=True, widths=0.3,
                         patch_artist=True,
                         boxprops=boxprops,
                         whiskerprops=whiskerprops,
                         medianprops=medianprops,
                         showmeans=showmeans,
                         meanprops=meanprops)
        
        # Окрашиваем коробки боксплота в светло-голубой с прозрачностью
        for box in bp['boxes']:
            box.set(facecolor='lightblue', alpha=0.6)
        
        # Выбросы отображаем красными полупрозрачными кружками
        for flier in bp['fliers']:
            flier.set(marker='o', color='red', alpha=0.5, markersize=4)
        
        # Оформление боксплота с подписями и поворотом меток по оси Х
        ax2.set_title('Boxplot', fontsize=11, fontweight='bold')
        ax2.set_ylabel('Значение', fontsize=10)
        ax2.set_xticks([1])
        ax2.set_xticklabels([field], fontsize=10, rotation=45)

        # Форматируем ось Y боксплота по диапазону значений
        if data_range < 0.1:
            formatter_box = FuncFormatter(lambda x, _: f"{x:.4f}")
        elif data_range > 10000:
            formatter_box = FuncFormatter(lambda x, _: f"{x/1000:.0f}K")
        else:
            formatter_box = FuncFormatter(lambda x, _: f"{x:.0f}" if x % 1 == 0 else f"{x:.1f}")
        
        ax2.yaxis.set_major_formatter(formatter_box)
        ax2.grid(True, linestyle=':', alpha=0.5, axis='y')
        
        # Вычисляем межквартильный размах и границы усов по правилу 1.5*IQR
        q1 = clean_data.quantile(0.25)
        q3 = clean_data.quantile(0.75)
        iqr = q3 - q1
        lower_whisker = max(clean_data.min(), q1 - 1.5 * iqr)
        upper_whisker = min(clean_data.max(), q3 + 1.5 * iqr)
        
        # Определяем выбросы вне усов
        outliers = clean_data[(clean_data < lower_whisker) | (clean_data > upper_whisker)]
        outlier_percent = len(outliers) / len(clean_data) * 100 if len(clean_data) > 0 else 0
        
        # Добавляем статистическое описание боксплота
        stats_box_text = (f'Q1: {q1:.2f}\n'
                          f'Q3: {q3:.2f}\n'
                          f'IQR: {iqr:.2f}\n'
                          f'Выбросы: {outlier_percent:.1f}%')
        
        ax2.text(0.98, 0.02, stats_box_text,
                 transform=ax2.transAxes,
                 fontsize=9,
                 verticalalignment='bottom',
                 horizontalalignment='right',
                 bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.3))
        
        # Добавляем заголовок всей фигуры
        fig.suptitle(title, fontsize=12, fontweight='bold', y=1.02)
        
        # Оптимизируем расположение элементов и отображаем график
        plt.tight_layout()
        plt.show()


In [19]:
# Функция визуализирует распределение категориальных признаков из DataFrame,
# строя для каждого столбца столбчатую диаграмму с различными настройками отображения.
def plot_categorical_features(data, categorical_cols=None, id_col='id', top_n=None, sort_desc=True,
                              figsize=(14, 5), max_plots_per_figure=6, descriptions=None, bar_color='steelblue',
                              show_percentage_inside=True, percentage_format='inside', label_rotation=45,
                              label_wrap_length=15):

    # Если список категориальных признаков не задан, автоматически определяем столбцы с типом object или category
    if categorical_cols is None:
        categorical_cols = data.select_dtypes(include=['object', 'category']).columns.tolist()
    
    # Если категориальных признаков не найдено, информируем пользователя и прекращаем выполнение
    if not categorical_cols:
        print("Категориальных признаков не найдено")
        return

    # Исключаем из списка колонку с идентификатором, если он есть среди категориальных
    if id_col in categorical_cols:
        categorical_cols.remove(id_col)
    
    n_features = len(categorical_cols)  # Всего категориальных признаков для визуализации
    
    # Вычисляем необходимое число фигур, чтобы разбить графики по количеству на одну фигуру
    num_figures = (n_features + max_plots_per_figure - 1) // max_plots_per_figure
    
    # Проходим по каждой фигуре с подмножеством категориальных признаков
    for fig_num in range(num_figures):
        start_idx = fig_num * max_plots_per_figure
        end_idx = min((fig_num + 1) * max_plots_per_figure, n_features)
        current_cols = categorical_cols[start_idx:end_idx]
        
        n_current = len(current_cols)
        
        # Вычисляем размер фигуры в зависимости от числа графиков для упаковки по вертикали
        fig_height = figsize[1] * max(1, n_current * 0.8)
        fig, axes = plt.subplots(n_current, 1, figsize=(figsize[0], fig_height))
        
        # Если всего один график, переводим axes в список для единообразной обработки
        if n_current == 1:
            axes = [axes]
        
        # Построение каждого графика столбик за столбиком
        for idx, (ax, col) in enumerate(zip(axes, current_cols)):
            # Проверяем наличие столбца в данных
            if col not in data.columns:
                print(f"Предупреждение: столбец '{col}' не найден, пропускаем")
                ax.axis('off')  # Выключаем оси для пустого графика
                continue
            
            # Группируем данные по уникальным значениям столбца с подсчётом количества в id_col,
            # если id_col задан и присутствует в данных, иначе простое value_counts
            if id_col in data.columns:
                grouped_data = data.groupby(col)[id_col].count()
            else:
                grouped_data = data[col].value_counts()
            
            total_count = grouped_data.sum()  # Общее количество для вычисления процентов
            
            # Если задан top_n, отбираем топ-N значений для визуализации
            if top_n is not None and len(grouped_data) > top_n:
                sorted_data = grouped_data.sort_values(ascending=False)
                grouped_data = sorted_data.head(top_n)
            
            # Сортируем по убыванию или возрастанию в зависимости от параметра
            if sort_desc:
                grouped_data = grouped_data.sort_values(ascending=False)
            
            # Вычисляем процентное соотношение для каждого значения
            percentages = (grouped_data.values / total_count * 100).round(1)
            
            # Получаем описание колонки для заголовка, если доступно
            if descriptions and col in descriptions:
                description = descriptions[col]
            else:
                description = col
            
            # Строим столбчатую диаграмму
            bars = ax.bar(range(len(grouped_data)), grouped_data.values, 
                         color=bar_color, edgecolor='#333333', alpha=0.7)
            
            # Добавляем аннотации к столбцам со значениями и/или процентами в указанном формате
            for bar, value, percentage in zip(bars, grouped_data.values, percentages):
                height = bar.get_height()
                x_pos = bar.get_x() + bar.get_width() / 2
                
                value_str = format_adaptive(value, None)
                
                if percentage_format == 'inside':
                    # Проценты внутри столбцов, если высота столбца достаточно велика
                    if height > max(grouped_data.values) * 0.1:
                        ax.text(x_pos, height * 0.5, f'{percentage:.1f}%', 
                               ha='center', va='center', fontsize=10,
                               color='white', fontweight='bold')
                    # Значения немного над столбцами
                    ax.text(x_pos, height * 1.02, value_str, 
                           ha='center', va='bottom', fontsize=10)
                
                elif percentage_format == 'above':
                    # Проценты и значения над столбцами
                    ax.text(x_pos, height * 1.02, f'{value_str} ({percentage:.1f}%)', 
                           ha='center', va='bottom', fontsize=10)
                
                elif percentage_format == 'both':
                    # Проценты внутри и значения над столбцами
                    if height > max(grouped_data.values) * 0.1:
                        ax.text(x_pos, height * 0.5, f'{percentage:.1f}%', 
                               ha='center', va='center', fontsize=10,
                               color='white', fontweight='bold')
                    ax.text(x_pos, height * 1.02, value_str, 
                           ha='center', va='bottom', fontsize=10)
            
            # Заголовок графика с описанием и числом уникальных значений
            ax.set_title(f'{description} (Всего: {total_count:,}, Уникальных: {data[col].nunique()})', 
                         fontsize=12, pad=12, loc='left')
            
            # Форматируем метки по оси X, обрезая длинные тексты
            labels = []
            for x in grouped_data.index:
                label_str = str(x)
                if len(label_str) > label_wrap_length:
                    wrapped_label = '\n'.join([label_str[i:i+label_wrap_length] 
                                              for i in range(0, len(label_str), label_wrap_length)])
                    labels.append(wrapped_label)
                else:
                    labels.append(label_str)
            
            # Устанавливаем метки по оси X с поворотом и выравниванием
            ax.set_xticks(range(len(grouped_data)))
            ax.set_xticklabels(labels, rotation=label_rotation, ha='right', fontsize=9)
            
            # Добавляем отступ снизу, если есть многострочные метки
            if any('\n' in label for label in labels):
                ax.tick_params(axis='x', pad=15)
            
            # Подпись оси Y и форматирование чисел
            ax.set_ylabel('Количество', fontsize=10)
            ax.yaxis.set_major_formatter(FuncFormatter(format_adaptive))
            
            # Отключаем верхнюю и правую рамки графика
            ax.spines['right'].set_visible(False)
            ax.spines['top'].set_visible(False)
            
            # Включаем сетку по оси Y с прозрачностью
            ax.grid(True, axis='y', alpha=0.2)
            
            # Настраиваем границы по оси X, чтобы столбцы не прилипали к краям
            ax.set_xlim(-0.5, len(grouped_data) - 0.5)
        
        # Общий заголовок для всех графиков на фигуре
        suptitle = 'Распределение категориальных признаков'
        
        plt.suptitle(suptitle, fontsize=14, y=0.98)
        
        # Оптимизируем компоновку и отображаем
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.subplots_adjust(hspace=0.4)
        plt.show()


In [20]:
# Функция визуализирует распределение дат по указанным столбцам DataFrame,
# создавая графики с временной агрегацией и опциональным сглаживанием, трендами и статистиками.
def plot_date_distribution(data, date_cols=None, id_col='id',
                           figsize=(14, 5), max_plots_per_figure=6,
                           date_format='%Y-%m-%d', descriptions=None,
                           line_color='steelblue', fill_color='steelblue',
                           alpha=0.7, grid_alpha=0.2,
                           freq='D', resample_method='count',
                           smooth_window=None, show_trend=True,
                           trend_color='red', trend_linewidth=2,
                           show_statistics=True, stat_fontsize=10):

    # Если даты не заданы явно, автоматически ищем столбцы с типом datetime
    if date_cols is None:
        date_cols = data.select_dtypes(include=['datetime64', 'datetime']).columns.tolist()
    
    # Если не нашлось datetime-столбцов, пробуем преобразовать первые 100 непустых значений
    # из строковых столбцов для выявления потенциальных дат
    if not date_cols:
        string_cols = data.select_dtypes(include=['object']).columns.tolist()
        for col in string_cols:
            try:
                sample = data[col].dropna().head(100)
                pd.to_datetime(sample, errors='raise')
                date_cols.append(col)
            except:
                continue
    
    # Если подходящих столбцов не найдено, выводим сообщение и завершаем
    if not date_cols:
        print("Колонок с датами не найдено")
        return
    
    # Исключаем id_col из списка дат, если он там есть
    if id_col in date_cols:
        date_cols.remove(id_col)
    
    n_features = len(date_cols)  # Количество колонок с датами
    
    # Вычисляем количество отдельных фигур для визуализации,
    # разбивая графики на группы по max_plots_per_figure
    num_figures = (n_features + max_plots_per_figure - 1) // max_plots_per_figure
    
    # Словарь с человекочитаемыми названиями интервалов агрегации
    freq_dict = {
        'D': 'день',
        'W': 'неделя',
        'M': 'месяц',
        'Q': 'квартал',
        'Y': 'год'
    }
    
    freq_name = freq_dict.get(freq, freq)
    
    # Проходим по каждой фигуре — группе столбцов дат
    for fig_num in range(num_figures):
        start_idx = fig_num * max_plots_per_figure
        end_idx = min((fig_num + 1) * max_plots_per_figure, n_features)
        current_cols = date_cols[start_idx:end_idx]
        
        n_current = len(current_cols)
        
        # Создаём фигуру с числом строк графиков по количеству дат в текущей группе
        fig, axes = plt.subplots(n_current, 1, figsize=(figsize[0], figsize[1] * n_current))
        
        # Если график один — преобразовываем оси в список для удобства
        if n_current == 1:
            axes = [axes]
        
        # Для каждого столбца с датами строим график распределения событий
        for idx, (ax, col) in enumerate(zip(axes, current_cols)):
            # Проверяем, что столбец существует в данных
            if col not in data.columns:
                print(f"Предупреждение: столбец '{col}' не найден, пропускаем")
                ax.axis('off')
                continue
            
            temp_df = data[[col]].copy()
            
            # Если столбец не в формате datetime, пытаемся преобразовать
            if not pd.api.types.is_datetime64_any_dtype(temp_df[col]):
                try:
                    temp_df[col] = pd.to_datetime(temp_df[col], errors='coerce')
                except Exception as e:
                    print(f"Ошибка преобразования колонки '{col}' в datetime: {e}")
                    ax.axis('off')
                    continue
            
            # Удаляем строки с некорректными датами
            temp_df = temp_df.dropna(subset=[col])
            
            # Если данных после очистки нет — пропускаем
            if temp_df.empty:
                print(f"Колонка '{col}' не содержит корректных дат")
                ax.axis('off')
                continue
            
            # Сортируем по дате
            temp_df = temp_df.sort_values(col)
            
            # Если есть id_col, строим временной ряд количества уникальных или всех событий по интервалам
            if id_col in data.columns:
                temp_df = data[[col, id_col]].dropna(subset=[col]).copy()
                temp_df[col] = pd.to_datetime(temp_df[col], errors='coerce')
                temp_df = temp_df.dropna(subset=[col])
                
                if resample_method == 'nunique':
                    time_series = temp_df.set_index(col).groupby(pd.Grouper(freq=freq))[id_col].nunique()
                else:
                    temp_df = temp_df.set_index(col)
                    time_series = temp_df[id_col].resample(freq).count()
            else:
                # Иначе считаем количество записей по интервалам времени
                temp_df = temp_df.set_index(col)
                time_series = temp_df.resample(freq).size()
            
            # Заполняем пропуски нулями
            time_series = time_series.fillna(0)
            
            # Опциональное сглаживание с помощью скользящего среднего
            if smooth_window and len(time_series) > smooth_window:
                smoothed = time_series.rolling(window=smooth_window, center=True).mean()
            
            dates = time_series.index
            values = time_series.values
            
            # Строим линейный график событий и заливаем область под кривой
            ax.plot(dates, values, color=line_color, linewidth=2, label='Факт')
            ax.fill_between(dates, 0, values, color=fill_color, alpha=alpha)
            
            # Отображаем скользящее среднее, если задано
            if smooth_window and len(time_series) > smooth_window:
                ax.plot(dates, smoothed, color='darkorange', linewidth=2, 
                        linestyle='--', label=f'Скользящее среднее ({smooth_window})')
            
            # Строим линейный тренд, если выбрано и данных достаточно
            if show_trend and len(values) > 1:
                x_numeric = np.arange(len(values))
                if len(x_numeric) > 1:
                    z = np.polyfit(x_numeric, values, 1)
                    p = np.poly1d(z)
                    trend_line = p(x_numeric)
                    first_date, last_date = dates[0], dates[-1]
                    first_value, last_value = trend_line[0], trend_line[-1]
                    
                    ax.plot([first_date, last_date], [first_value, last_value], 
                            color=trend_color, linewidth=trend_linewidth, 
                            linestyle='-', label='Тренд')
            
            # Описание колонки для заголовка, если доступно
            if descriptions and col in descriptions:
                description = descriptions[col]
            else:
                description = col
            
            # Форматируем метки по оси X с заданным форматом дат
            ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter(date_format))
            plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right')
            
            # Подпись по оси Y в зависимости от метода агрегации
            if resample_method == 'nunique':
                ylabel = f'Количество уникальных значений ({freq_name})'
            else:
                ylabel = f'Количество событий ({freq_name})'
            ax.set_ylabel(ylabel, fontsize=10)
            
            # Форматирование подписей по оси Y с разделителями тысяч
            ax.yaxis.set_major_formatter(FuncFormatter(lambda x, p: format(int(x), ',')))
            
            # Заголовок графика с распределением по времени
            title_text = f'{description} - Распределение по времени'
            
            # Если нужно, добавляем статистическую информацию по ряду
            if show_statistics and len(values) > 0:
                stats_text = []
                stats_text.append(f"Период: {dates[0].strftime(date_format)} - {dates[-1].strftime(date_format)}")
                stats_text.append(f"Всего событий: {int(values.sum()):,}")
                stats_text.append(f"Среднее в {freq_name}: {values.mean():.1f}")
                stats_text.append(f"Максимум в {freq_name}: {values.max():.0f}")
                stats_text.append(f"Минимум в {freq_name}: {values.min():.0f}")
                
                if len(values) > 1:
                    change_pct = ((values[-1] - values[0]) / values[0] * 100) if values[0] != 0 else 0
                    stats_text.append(f"Изменение: {change_pct:+.1f}%")
                
                stats_str = " | ".join(stats_text)
                ax.set_title(f'{title_text}\n{stats_str}', fontsize=11, pad=15, loc='left')
            else:
                ax.set_title(title_text, fontsize=12, pad=12, loc='left')
            
            # Отображаем легенду при наличии сглаживания или тренда
            if smooth_window or show_trend:
                ax.legend(loc='upper left', fontsize=9)
            
            # Включаем сетку и отключаем верхнюю и правую рамки графика
            ax.grid(True, alpha=grid_alpha, linestyle='--')
            ax.spines['right'].set_visible(False)
            ax.spines['top'].set_visible(False)
        
        # Общий заголовок для всей фигуры с графиками
        suptitle = f'Распределение событий по датам (агрегация: {freq_name})'
        plt.suptitle(suptitle, fontsize=14, y=0.98)
        
        plt.tight_layout(rect=[0, 0, 1, 0.96])
        plt.show()


In [21]:
# Функция визуализирует важность признаков на основе переданных данных feature_importance,
# используя словарь описаний fields_description для подстановки человекочитаемых меток.
def plot_permutation_importance(feature_importance, fields_description, title='Важность признаков для модели', 
                                figsize=(12, 8), text_offset=0.003, fontsize=10, grid_alpha=0.5):
    # Создаём копию DataFrame с важностями, чтобы не менять исходный объект
    fi = feature_importance.copy()
    
    # Вспомогательная функция для получения метки признака из словаря описаний или оригинального названия
    def get_label(text):
        return fields_description.get(text, text)
    
    # Добавляем столбец с метками признаков
    fi['feature_label'] = fi['feature'].apply(get_label)
    # Сортируем по важности признаков в порядке убывания
    fi = fi.sort_values('importance', ascending=False)
    
    n_features = len(fi)
    # Расчёт высоты фигуры в зависимости от числа признаков (мин. 6 дюймов)
    fig_height = max(6, n_features * 0.5)
    plt.figure(figsize=(figsize[0], fig_height))
    
    # Создаём горизонтальный barplot важностей с единым цветом
    ax = sns.barplot(
        x='importance', 
        y='feature_label', 
        data=fi, 
        color='#4E79A7'
    )
    
    # Подписываем каждый столбец значением важности справа от бара
    for i, (_, row) in enumerate(fi.iterrows()):
        ax.text(
            row['importance'] + text_offset,  # Немного правее края столбца
            i, 
            f"{row['importance']:.3f}",       # Форматируем число с тремя знаками после запятой
            va='center',                      # Вертикальное выравнивание по центру
            ha='left',                       # Горизонтальное выравнивание слева от текста
            fontsize=fontsize,
            bbox=dict(facecolor='white', alpha=0.7, edgecolor='none', pad=0.3)  # Легкий фон для читаемости
        )
    
    # Заголовок и подписи осей с заданным размером и цветом шрифта
    plt.title(title, fontsize=14, pad=12, color='#333333')
    plt.xlabel('Значимость признака', fontsize=12, color='#333333')
    plt.ylabel('')
    
    # Устанавливаем пределы по оси X с небольшим запасом сверху
    xmax = fi['importance'].max() * 1.1
    plt.xlim(0, xmax)
    
    # Включаем пунктирную сетку с заданной прозрачностью и цветом
    plt.grid(True, linestyle=':', alpha=grid_alpha, color='#dddddd')
    
    # Устанавливаем размер шрифта меток по оси Y
    plt.yticks(fontsize=fontsize)
    
    # Оптимизируем компоновку для аккуратного отображения
    plt.tight_layout()
    
    # Увеличиваем отступ слева, чтобы длинные подписи не обрезались
    plt.subplots_adjust(left=0.4)
    
    # Отображаем итоговый график
    plt.show()


In [22]:
# Функция строит тепловую карту (heatmap) для сгруппированных данных с возможностью указать описания полей и настройки оформления.
def draw_heatmap_by_grouped_data(data, fields_description=None, title='', xlabel='', ylabel='', 
                                figsize=None, cmap="YlOrRd", cell_size=2):
    # Автоматический расчет размера фигуры, если figsize не передан
    if figsize is None:
        n_rows, n_cols = data.shape
        fig_width = n_cols * cell_size + 2  # Ширина с запасом на отступы
        fig_height = n_rows * cell_size + 1  # Высота с запасом
        figsize = (fig_width, fig_height)
    
    # Форматируем значения ячеек с помощью функции format_adaptive для удобочитаемого отображения
    annotations = data.applymap(format_adaptive)
    
    # Создаем новую фигуру с заданным размером
    plt.figure(figsize=figsize)
    
    # Рисуем тепловую карту с аннотациями и заданной цветовой палитрой
    ax = sns.heatmap(
        data,
        annot=annotations,            # Подписи в ячейках с форматированием
        fmt="s",                     # Формат подписей — строковый
        cmap=cmap,                   # Цветовая карта
        linewidths=0.5,              # Толщина линий между ячейками
        linecolor="lightgray",       # Цвет разделительных линий
        cbar_kws={                   # Настройки цветовой шкалы справа
            'format': plt.FuncFormatter(format_adaptive),  # Форматирование подписей цветовой шкалы
            'shrink': 0.7},           # Сжатие полосы цветовой шкалы
        vmin=data.min().min(),       # Минимальное значение для цветового диапазона
        vmax=data.max().max(),       # Максимальное значение для цветового диапазона
        square=True                  # Делает ячейки квадратными
    )
    
    # Если передан словарь с описаниями полей, заменяем метки на человекочитаемые
    if fields_description is not None:
        # Обновляем подписи оси Y
        row_labels = [fields_description.get(str(idx), str(idx)) for idx in data.index]
        ax.set_yticklabels(row_labels, rotation=0, ha='right', va='center')
        
        # Обновляем подписи оси X
        col_labels = [fields_description.get(str(col), str(col)) for col in data.columns]
        ax.set_xticklabels(col_labels, rotation=45, ha='right')
        
    # Устанавливаем заголовок и подписи осей с заданным размером шрифта и отступами
    plt.title(title, pad=20, fontsize=14)
    plt.xlabel(xlabel, fontsize=12)
    plt.ylabel(ylabel, fontsize=12)
    
    # Оптимизируем компоновку графика по элементам, чтобы избежать наложений
    plt.tight_layout()
    
    # Подстраиваем отступы с учетом количества строк, чтобы метки осей не обрезались
    ax.figure.subplots_adjust(
        left=max(0.2, 0.3 - 0.01 * len(data.index)),
        bottom=0.2,
        right=0.9
    )
    
    # Отображаем тепловую карту
    plt.show()


In [23]:
# Функция находит пары признаков с высокой абсолютной корреляцией, превышающей заданный порог.
def get_high_correlations(corr_matrix, threshold=0.5):
    # Создаём копию корреляционной матрицы, чтобы не модифицировать исходную
    corr = corr_matrix.copy()
    # Заменяем диагональные элементы на NaN, чтобы игнорировать автокорреляцию (корреляцию признака сам с собой)
    np.fill_diagonal(corr.values, np.nan)
    
    results = []  # Список для хранения результатов
    features = corr.columns  # Список признаков (названия столбцов)
    
    # Проходим по верхнему треугольнику матрицы корреляций, без диагонали
    for i in range(len(features)):
        for j in range(i+1, len(features)):
            corr_value = corr.iloc[i, j]  # Значение корреляции между признаками i и j
            # Проверяем, что значение не NaN и по модулю превышает порог
            if pd.notna(corr_value) and abs(corr_value) >= threshold:
                results.append({
                    'Признак 1': features[i],
                    'Признак 2': features[j],
                    'Корреляция': corr_value
                })
    
    # Преобразуем список словарей в DataFrame
    result_df = pd.DataFrame(results)
    
    # Если результат не пуст, сортируем по убыванию абсолютного значения корреляции
    if not result_df.empty:
        result_df = result_df.sort_values('Корреляция', key=abs, ascending=False)
    
    # Возвращаем DataFrame с найденными высокими корреляциями
    return result_df


In [24]:
# Функция строит barplot важности признаков по методу SHAP с дополнительными настройками отображения и описаниями.
def plot_shap_bar(shap_values, 
                 fields_description=None,
                 title="Важность признаков по методу SHAP",
                 xlabel="Среднее абсолютное значение SHAP",
                 ylabel="Признаки",
                 explanation_text="SHAP (SHapley Additive exPlanations) - метод объяснения предсказаний моделей машинного обучения.\nЗначения показывают вклад каждого признака в итоговое предсказание модели.",
                 max_display=17,
                 title_fontsize=14,
                 label_fontsize=12,
                 value_fontsize=10,
                 explanation_fontsize=11,
                 bbox_facecolor="orange",
                 bbox_alpha=0.1,
                 bbox_pad=5,
                 fig_width=12,
                 base_height=0.5,
                 per_feature_height=0.4,
                 show=True):

    # Вычисляем высоту фигуры с учётом количества отображаемых признаков
    height = base_height + max_display * per_feature_height
    plt.figure(figsize=(fig_width, height))
    
    # Если есть словарь описаний, заменяем имена признаков для удобства интерпретации
    if fields_description:
        feature_names = shap_values.feature_names
        translated_names = []
        for name in feature_names:
            # Убираем префиксы в формате "xxx__name", если есть
            clean_name = name.split('__')[-1] if '__' in name else name
            translated_names.append(fields_description.get(clean_name, name))
        
        # Создаём новый объект Explanation с заменёнными именами признаков
        shap_values = shap.Explanation(
            values=shap_values.values,
            base_values=shap_values.base_values,
            data=shap_values.data,
            feature_names=translated_names
        )
    
    # Построение barplot SHAP-значений с ограничением по количеству признаков, без немедленного отображения
    shap.summary_plot(
        shap_values, 
        plot_type="bar", 
        max_display=max_display,
        show=False
    )
    
    ax = plt.gca()
    
    # Устанавливаем заголовок и подписи осей
    plt.title(title, fontsize=title_fontsize)
    plt.xlabel(xlabel, fontsize=label_fontsize)
    plt.ylabel(ylabel, fontsize=label_fontsize)
    
    # Добавляем цифровые подписи значений SHAP справа от каждого бара
    for bar in ax.containers[0]:
        width = bar.get_width()
        ax.annotate(f"{width:.4f}", 
                    (width, bar.get_y() + bar.get_height() / 2),
                    xytext=(5, 0),  # Смещение текста вправо
                    textcoords="offset points",
                    va='center',
                    fontsize=value_fontsize)
        
    # Добавляем пояснительный текст под графиком с фоном и прозрачностью
    plt.figtext(0.5, 0.01, explanation_text, 
                ha="center", 
                fontsize=explanation_fontsize,
                bbox={"facecolor": bbox_facecolor, "alpha": bbox_alpha, "pad": bbox_pad})
    
    # Настраиваем отступы для корректного отображения элементов
    plt.subplots_adjust(
        left=0.3,
        right=0.95,
        bottom=0.15 + len(explanation_text.split('\n')) * 0.05,  # Учитываем высоту текста
        top=0.9
    )
    
    # Показываем график, если параметр show установлен в True
    if show:
        plt.show()


In [25]:
# Функция строит диаграмму рассеяния SHAP (beeswarm plot),
# отображающую влияние признаков на предсказания модели с возможностью замены подписей по признакам.
def plot_shap_beeswarm(shap_values, 
                       features,
                       fields_description=None,
                       title="Диаграмма рассеяния SHAP",
                       xlabel="Значение SHAP",
                       ylabel="Признаки",
                       explanation_text="Диаграмма рассеяния SHAP показывает распределение влияния признаков на модель.\nКаждая точка представляет одно наблюдение. Цвет показывает значение признака.",
                       max_display=17,
                       title_fontsize=14,
                       label_fontsize=12,
                       explanation_fontsize=11,
                       bbox_facecolor="lightblue",
                       bbox_alpha=0.1,
                       bbox_pad=5,
                       fig_width=12,
                       base_height=0.5,
                       per_feature_height=0.4,
                       show=True):
    
    # Расчёт высоты фигуры пропорционально числу отображаемых признаков
    height = base_height + max_display * per_feature_height
    plt.figure(figsize=(fig_width, height))
    
    # Если есть словарь описаний, заменяем имена признаков на читаемые
    if fields_description:
        feature_names = shap_values.feature_names
        translated_names = []
        for name in feature_names:
            # Убираем возможные префиксы в названии признака
            clean_name = name.split('__')[-1] if '__' in name else name
            translated_names.append(fields_description.get(clean_name, name))
        
        # Создаем новый объект SHAP Explanation с обновленными именами признаков
        shap_values = shap.Explanation(
            values=shap_values.values,
            base_values=shap_values.base_values,
            data=shap_values.data,
            feature_names=translated_names
        )
    
    # Построение beeswarm диаграммы с максимальным числом признаков, без автоматического отображения
    shap.summary_plot(
        shap_values, 
        features=features,
        plot_type="dot",
        max_display=max_display,
        show=False
    )
    
    # Получаем текущие оси для дальнейшей настройки
    ax = plt.gca()
    
    # Устанавливаем заголовок и подписи осей с заданным размером шрифта
    plt.title(title, fontsize=title_fontsize)
    plt.xlabel(xlabel, fontsize=label_fontsize)
    plt.ylabel(ylabel, fontsize=label_fontsize)
    
    # Добавляем пояснительный текст под графиком с цветным фоном и прозрачностью
    plt.figtext(0.5, 0.01, explanation_text, 
                ha="center", 
                fontsize=explanation_fontsize,
                bbox={"facecolor": bbox_facecolor, "alpha": bbox_alpha, "pad": bbox_pad})
    
    # Настраиваем отступы, учитывая длину пояснительного текста
    plt.subplots_adjust(
        left=0.25,
        right=0.95,
        bottom=0.15 + len(explanation_text.split('\n')) * 0.05,
        top=0.9
    )
    
    # Пытаемся получить цветовую шкалу из элементов текущего рисунка
    try:
        color_bar = plt.gcf().axes[-1]
        color_bar.set_ylabel("Значение признака", fontsize=label_fontsize)
    except (IndexError, AttributeError):
        # Если шкалы нет, создаём её вручную на основе диапазона значений признаков
        norm = plt.Normalize(
            features.min().min(), 
            features.max().max()
        )
        sm = plt.cm.ScalarMappable(cmap="coolwarm", norm=norm)
        sm.set_array([])
        color_bar = plt.colorbar(sm, ax=ax)
        color_bar.set_label("Значение признака", fontsize=label_fontsize)
    
    # Отображаем построенный график, если параметр show установлен в True
    if show:
        plt.show()


In [26]:
# Функция строит матрицу ошибок (confusion matrix) с числовыми значениями и процентами,
# а также аннотациями для бинарной классификации.
def plot_confusion_matrix(y_true, y_pred, class_names=None, figsize=(10, 8)):

    # Вычисляем матрицу ошибок на основе истинных и предсказанных значений
    cm = confusion_matrix(y_true, y_pred)
    
    # Если названия классов не переданы, задаём по умолчанию для бинарной задачи
    if class_names is None:
        class_names = ['0', '1']
    
    # Вычисляем процентное соотношение ошибок в каждой строке
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    # Создаём фигуру и оси с указанным размером
    fig, ax = plt.subplots(figsize=figsize)

    # Отображаем матрицу ошибок в виде цветной карты с использованием палитры 'Blues'
    im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
    # Добавляем цветовую шкалу справа
    ax.figure.colorbar(im, ax=ax)
    
    # Задаём порог для выбора цвета текста, чтобы обеспечить читаемость (белый на тёмном фоне и чёрный на светлом)
    thresh = cm.max() / 2.
    
    # Подписываем каждую ячейку количеством ошибок и процентом
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i,
                f'{cm[i, j]}\n({cm_percent[i, j]:.1f}%)',  # Текст: число и процент
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black",
                fontweight='bold', fontsize=12
            )
    
    # Настраиваем оси: метки, подписи и заголовок
    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=class_names,
        yticklabels=class_names,
        ylabel='Фактические значения',
        xlabel='Предсказанные значения',
        title='Матрица ошибок с процентами'
    )
    
    # Поворачиваем подписи по оси X для лучшей читаемости
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")
    
    # Для бинарной классификации добавляем крупные обозначения TN, FP, FN, TP
    if cm.shape == (2, 2):
        y_offset = 0.25  # Немного поднимаем надписи над ячейками
        ax.text(0, 0 + y_offset, 'TN', ha='center', va='bottom',
                fontweight='bold', fontsize=36, color='black', alpha=0.8)
        ax.text(1, 0 + y_offset, 'FP', ha='center', va='bottom',
                fontweight='bold', fontsize=36, color='black', alpha=0.8)
        ax.text(0, 1 + y_offset, 'FN', ha='center', va='bottom',
                fontweight='bold', fontsize=36, color='black', alpha=0.8)
        ax.text(1, 1 + y_offset, 'TP', ha='center', va='bottom',
                fontweight='bold', fontsize=36, color='black', alpha=0.8)
    
    # Оптимизируем расположение элементов на рисунке
    plt.tight_layout()
    # Показываем итоговый график
    plt.show()


In [27]:
def data_review(df: pd.DataFrame) -> None:
    """
    Выводит сводную информацию о переданном DataFrame.

    Параметры:
    -----------
    df : pd.DataFrame
        Исходный DataFrame для обзора.

    Функция выполняет:
    - вывод первых 5 строк таблицы,
    - отображение информации о столбцах, типах и количестве не-null значений,
    - статистическое описание всех столбцов (включая категориальные).
    """
    # Вывод заголовка и первых пяти строк данных
    display(Markdown('##### Первые 5 строк датасета'))
    display(df.head())

    # Вывод заголовка и информации по структуре DataFrame
    display(Markdown('##### Информация о столбцах и типах данных'))
    print(df.info())

    # Вывод заголовка и статистическое описание для всех столбцов
    display(Markdown('##### Статистическое описание'))
    display(df.describe(include='all'))

### Загрузка данных

In [28]:
# Загружаем данные из CSV-файла 'apparel-messages.csv' в DataFrame messages
purchases = pd.read_csv('./filtered_data/apparel-purchases.csv')


In [29]:
# Загружаем данные о рассылках из CSV-файла по указанному локальному пути в DataFrame messages
messages = pd.read_csv('./filtered_data/apparel-messages.csv')


In [30]:
# Загрузка целевого набора данных из CSV-файла с указанного локального пути в DataFrame target
target = pd.read_csv('./filtered_data/apparel-target_binary.csv')


#### Просмотр данных

In [31]:
data_review(purchases)

##### Первые 5 строк датасета

,client_id,quantity,price,category_ids,date,message_id
0,1515915625468169594,1,1999.0,"['4', '28', '57', '431']",2022-05-16,1515915625468169594-4301-627b661e9736d
1,1515915625468169594,1,2499.0,"['4', '28', '57', '431']",2022-05-16,1515915625468169594-4301-627b661e9736d
2,1515915625471138230,1,6499.0,"['4', '28', '57', '431']",2022-05-16,1515915625471138230-4437-6282242f27843
3,1515915625471138230,1,4999.0,"['4', '28', '244', '432']",2022-05-16,1515915625471138230-4437-6282242f27843
4,1515915625471138230,1,4999.0,"['4', '28', '49', '413']",2022-05-16,1515915625471138230-4437-6282242f27843


##### Информация о столбцах и типах данных

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202208 entries, 0 to 202207
Data columns (total 6 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   client_id     202208 non-null  int64  
 1   quantity      202208 non-null  int64  
 2   price         202208 non-null  float64
 3   category_ids  202208 non-null  object 
 4   date          202208 non-null  object 
 5   message_id    202208 non-null  object 
dtypes: float64(1), int64(2), object(3)
memory usage: 9.3+ MB
None


##### Статистическое описание

,client_id,quantity,price,category_ids,date,message_id
count,2.022080e+05,202208.000000,202208.000000,202208,202208,202208
unique,NaN,NaN,NaN,933,642,50204
top,NaN,NaN,NaN,"['4', '28', '57', '431']",2022-11-11,1515915625489095763-6251-6311b13a4cf78
freq,NaN,NaN,NaN,8626,5270,365
mean,1.515916e+18,1.006483,1193.301516,NaN,NaN,NaN
std,1.459514e+08,0.184384,1342.252664,NaN,NaN,NaN
min,1.515916e+18,1.000000,1.000000,NaN,NaN,NaN
25%,1.515916e+18,1.000000,352.000000,NaN,NaN,NaN
50%,1.515916e+18,1.000000,987.000000,NaN,NaN,NaN
75%,1.515916e+18,1.000000,1699.000000,NaN,NaN,NaN


In [32]:
data_review(messages)

##### Первые 5 строк датасета

,bulk_campaign_id,client_id,message_id,event,channel,date,created_at
0,4439,1515915625626736623,1515915625626736623-4439-6283415ac07ea,open,email,2022-05-19,2022-05-19 00:14:20
1,4439,1515915625490086521,1515915625490086521-4439-62834150016dd,open,email,2022-05-19,2022-05-19 00:39:34
2,4439,1515915625553578558,1515915625553578558-4439-6283415b36b4f,open,email,2022-05-19,2022-05-19 00:51:49
3,4439,1515915625553578558,1515915625553578558-4439-6283415b36b4f,click,email,2022-05-19,2022-05-19 00:52:20
4,4439,1515915625471518311,1515915625471518311-4439-628341570c133,open,email,2022-05-19,2022-05-19 00:56:52


##### Информация о столбцах и типах данных

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12739798 entries, 0 to 12739797
Data columns (total 7 columns):
 #   Column            Dtype 
---  ------            ----- 
 0   bulk_campaign_id  int64 
 1   client_id         int64 
 2   message_id        object
 3   event             object
 4   channel           object
 5   date              object
 6   created_at        object
dtypes: int64(2), object(5)
memory usage: 680.4+ MB
None


##### Статистическое описание

,bulk_campaign_id,client_id,message_id,event,channel,date,created_at
count,1.273980e+07,1.273980e+07,12739798,12739798,12739798,12739798,12739798
unique,NaN,NaN,9061667,11,2,638,4103539
top,NaN,NaN,1515915625489095763-6251-6311b13a4cf78,send,mobile_push,2023-06-10,2023-12-29 15:20:53
freq,NaN,NaN,1454,9058196,7512156,89661,621
mean,1.160459e+04,1.515916e+18,NaN,NaN,NaN,NaN,NaN
std,3.259211e+03,3.265518e+08,NaN,NaN,NaN,NaN,NaN
min,5.480000e+02,1.515916e+18,NaN,NaN,NaN,NaN,NaN
25%,8.746000e+03,1.515916e+18,NaN,NaN,NaN,NaN,NaN
50%,1.351600e+04,1.515916e+18,NaN,NaN,NaN,NaN,NaN
75%,1.415800e+04,1.515916e+18,NaN,NaN,NaN,NaN,NaN


In [33]:
data_review(target)

##### Первые 5 строк датасета

,client_id,target
0,1515915625468060902,0
1,1515915625468061003,1
2,1515915625468061099,0
3,1515915625468061100,0
4,1515915625468061170,0


##### Информация о столбцах и типах данных

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49849 entries, 0 to 49848
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   client_id  49849 non-null  int64
 1   target     49849 non-null  int64
dtypes: int64(2)
memory usage: 779.0 KB
None


##### Статистическое описание

,client_id,target
count,4.984900e+04,49849.000000
mean,1.515916e+18,0.019278
std,1.487947e+08,0.137503
min,1.515916e+18,0.000000
25%,1.515916e+18,0.000000
50%,1.515916e+18,0.000000
75%,1.515916e+18,0.000000
max,1.515916e+18,1.000000


**Выводы:**

Загруженные наборы данных полностью соответствуют требованиям и характеристикам, заявленным в постановке задачи.

В процессе первичного обследования данных пропущенные значения отсутствуют, что свидетельствует о полноте информации для дальнейшего анализа.

## Предобработка данных

### Переименуем столбцы

In [34]:
# Переименовываем столбцы DataFrame messages для удобства понимания и единообразия
messages = messages.rename(columns={
    'date': 'message_date',
    'created_at': 'message_created_at'
})

In [35]:
# Переименование столбца 'date' в 'purchase_date' для ясности и однозначности в DataFrame purchases
purchases = purchases.rename(columns={
    'date': 'purchase_date'
})


### Сформируем словарь

In [36]:
# Словарь с описаниями полей, используемый для удобства интерпретации признаков в данных
fields_description = {
    'bulk_campaign_id': 'идентификатор рассылки', 
    'client_id': 'идентификатор клиента', 
    'message_id': 'идентификатор сообщения из рассылки', 
    'event': 'действие с сообщением', 
    'channel': 'канал рассылки',
    'message_date': 'дата действия рассылки', 
    'message_created_at': 'дата создания рассылки',

    'quantity': 'количество единиц товара', 
    'price': 'цена товара', 
    'category_ids': 'идентификаторы категорий', 
    'purchase_date': 'дата покупки', 

    'target': 'клиент совершил покупку в целевом периоде (целевая переменная)'
}

### Проверка типов данных

#### Данные о покупках клиентов по дням и по товарам.

In [37]:
# Выводит информацию о DataFrame messages: количество непустых значений, типы данных и использование памяти
purchases.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 202208 entries, 0 to 202207
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   client_id      202208 non-null  int64  
 1   quantity       202208 non-null  int64  
 2   price          202208 non-null  float64
 3   category_ids   202208 non-null  object 
 4   purchase_date  202208 non-null  object 
 5   message_id     202208 non-null  object 
dtypes: float64(1), int64(2), object(3)
memory usage: 9.3+ MB


##### client_id — идентификатор клиента (int64)

In [38]:
# Выводит количество записей в столбце 'client_id' DataFrame purchases,
# которые содержат числовые значения с дробной частью
print_numeric_with_fraction_count(purchases, 'client_id')


Количество записей с дробной частью: 0


##### message_id — идентификатор сообщения из рассылки (object)

In [39]:
# Вывод общего числа строк (записей) в DataFrame purchases
print('Общее количество строк: ', purchases.shape[0])

# Вывод количества уникальных значений в столбце 'message_id' DataFrame purchases
print('Количество уникальных message_id: ', purchases['message_id'].nunique())


Общее количество строк:  202208
Количество уникальных message_id:  50204


##### quantity — количество единиц товара (int64)

In [40]:
# Вывод количества записей в столбце 'quantity', содержащих значения с дробной частью
print_numeric_with_fraction_count(purchases, 'quantity')

# Вывод минимального значения количества товара в столбце 'quantity'
print('Минимальное количество товара: ', purchases['quantity'].min())

# Вывод максимального значения количества товара в столбце 'quantity'
print('Максимальное количество товара: ', purchases['quantity'].max())


Количество записей с дробной частью: 0
Минимальное количество товара:  1
Максимальное количество товара:  30


##### price — цена товара (float64 -> int64)

In [41]:
# Выводит количество записей в столбце 'price' DataFrame purchases,
# которые содержат значения с дробной частью (десятичные значения)
print_numeric_with_fraction_count(purchases, 'price')


Количество записей с дробной частью: 0


In [42]:
# Преобразуем тип данных столбца 'price' в DataFrame purchases в целочисленный (int64),
# так как цена в данных рассматривается как целое число без десятичных частей
purchases['price'] = purchases['price'].astype('int64')


In [43]:
# Вывод минимальной цены товара в столбце 'price' DataFrame purchases
print('Минимальная цена товара: ', purchases['price'].min())

# Вывод максимальной цены товара в столбце 'price' DataFrame purchases
print('Максимальная цена товара: ', purchases['price'].max())


Минимальная цена товара:  1
Максимальная цена товара:  85499


Преобразуем тип данных из float64 в int64, так как данный показатель является ценой, которая не имеет дробной части.

##### category_ids — идентификаторы категорий (object)

In [44]:
# Подсчитываем количество вхождений каждого уникального значения в столбце 'category_ids' DataFrame purchases,
# что позволяет выявить частоту появления каждой категории товара
purchases['category_ids'].value_counts()


category_ids
['4', '28', '57', '431']            8626
['4', '28', '260', '420']           6989
['4', '28', '244', '432']           6821
[]                                  5579
['4', '28', '275', '421']           4936
['2', '18', '258', '441']           4905
['4', '28', '62', '657']            4708
['4', '28', '62', '656']            3967
['4', '28', '124', '415']           3791
['4', '28', '275', '673']           3276
['4', '28', '213', '436']           3245
['4', '28', '343', '425']           3083
['4', '28', '290', '422']           2987
['4', '31', '326', '505']           2843
['2', '18', '344', '445']           2314
['4', '28', '249', '616']           2242
['2', '18', '61', '661']            2133
['4', '28', '146', '548']           2078
['4', '28', '58', '434']            2046
['4', '31', '324', '466']           1904
['4', '28', '104', '429']           1903
['4', '28', '49', '413']            1850
['4', '28', '249', '615']           1838
['2', '18', '212', '726']           1804
['5

##### purchase_date — дата покупки (object)

In [45]:
# Вывод общего количества строк (записей) в DataFrame purchases
print('Общее количество строк: ', purchases.shape[0])

# Вывод количества уникальных дат покупок в столбце 'purchase_date'
print('Количество уникальных purchase_date: ', purchases['purchase_date'].nunique())

# Вывод минимальной (ранней) даты покупки
print('Минимальная purchase_date: ', purchases['purchase_date'].min())

# Вывод максимальной (поздней) даты покупки
print('Максимальная purchase_date: ', purchases['purchase_date'].max())


Общее количество строк:  202208
Количество уникальных purchase_date:  642
Минимальная purchase_date:  2022-05-16
Максимальная purchase_date:  2024-02-16


#### Данные о рассылках, которые были отправлены клиентам из таблицы покупок.

In [46]:
# Выводит сводную информацию о DataFrame messages, включая количество непустых значений в каждом столбце
messages.info(show_counts=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12739798 entries, 0 to 12739797
Data columns (total 7 columns):
 #   Column              Non-Null Count     Dtype 
---  ------              --------------     ----- 
 0   bulk_campaign_id    12739798 non-null  int64 
 1   client_id           12739798 non-null  int64 
 2   message_id          12739798 non-null  object
 3   event               12739798 non-null  object
 4   channel             12739798 non-null  object
 5   message_date        12739798 non-null  object
 6   message_created_at  12739798 non-null  object
dtypes: int64(2), object(5)
memory usage: 680.4+ MB


##### client_id — идентификатор клиента (int64)

In [47]:
# Выводит количество записей в столбце 'client_id' DataFrame messages,
# где значения содержат дробную часть (то есть не являются целыми числами)
print_numeric_with_fraction_count(messages, 'client_id')


Количество записей с дробной частью: 0


##### bulk_campaign_id — идентификатор рассылки (int64)

In [48]:
# Выводит количество записей в столбце 'bulk_campaign_id' DataFrame messages,
# которые содержат значения с дробной частью (десятичные числа)
print_numeric_with_fraction_count(messages, 'bulk_campaign_id')


Количество записей с дробной частью: 0


##### message_id — идентификатор сообщения из рассылки (int64)

In [49]:
# Вывод общего количества строк (записей) в DataFrame messages
print('Общее количество строк: ', messages.shape[0])

# Вывод количества уникальных значений в столбце 'message_id' DataFrame messages
print('Количество уникальных message_id: ', messages['message_id'].nunique())


Общее количество строк:  12739798
Количество уникальных message_id:  9061667


##### event — действие с сообщением (object)

In [50]:
# Подсчитываем количество появлений каждого уникального значения в столбце 'event' DataFrame messages,
# что позволяет узнать частоту различных типов событий
messages['event'].value_counts()


event
send           9058196
open           3085820
click           496339
purchase         64679
hard_bounce      19903
soft_bounce      10583
unsubscribe       2841
hbq_spam           823
complain           528
subscribe           85
close                1
Name: count, dtype: int64

##### channel — канал рассылки (object)

In [51]:
# Подсчитываем количество каждого уникального значения в столбце 'channel' DataFrame messages,
# чтобы оценить распределение каналов рассылки
messages['channel'].value_counts()


channel
mobile_push    7512156
email          5227642
Name: count, dtype: int64

##### message_date — дата действия рассылки (object)

In [52]:
# Выводит общее количество строк (записей) в DataFrame messages
print('Общее количество строк: ', messages.shape[0])

# Выводит количество уникальных дат в столбце 'message_date'
print('Количество уникальных message_date: ', messages['message_date'].nunique())

# Выводит минимальное значение даты в столбце 'message_date'
print('Минимальная message_date: ', messages['message_date'].min())

# Выводит максимальное значение даты в столбце 'message_date'
print('Максимальная message_date: ', messages['message_date'].max())


Общее количество строк:  12739798
Количество уникальных message_date:  638
Минимальная message_date:  2022-05-19
Максимальная message_date:  2024-02-15


##### message_created_at — дата создания рассылки (object)

In [53]:
# Вывод общего количества строк в DataFrame messages
print('Общее количество строк: ', messages.shape[0])

# Вывод количества уникальных значений в столбце 'message_created_at'
print('Количество уникальных message_created_at: ', messages['message_created_at'].nunique())

# Вывод минимального значения даты и времени в столбце 'message_created_at'
print('Минимальная message_created_at: ', messages['message_created_at'].min())

# Вывод максимального значения даты и времени в столбце 'message_created_at'
print('Максимальная message_created_at: ', messages['message_created_at'].max())



Общее количество строк:  12739798
Количество уникальных message_created_at:  4103539
Минимальная message_created_at:  2022-05-19 00:14:20
Максимальная message_created_at:  2024-02-15 23:58:40


#### Данные о совершении клиентом покупки в целевом периоде (целевой показатель).

In [54]:
# Выводит краткую информацию о DataFrame target:
# количество непустых значений, типы данных по столбцам и использование памяти
target.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49849 entries, 0 to 49848
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   client_id  49849 non-null  int64
 1   target     49849 non-null  int64
dtypes: int64(2)
memory usage: 779.0 KB


##### client_id — идентификатор клиента (int64)

In [55]:
# Выводит количество записей в столбце 'client_id' DataFrame target,
# в которых значения имеют дробную часть (то есть не являются целыми числами)
print_numeric_with_fraction_count(target, 'client_id')


Количество записей с дробной частью: 0


##### taget — клиент совершил покупку в целевом периоде (целевая переменная) (int64)

In [56]:
# Подсчитываем количество каждого уникального значения в столбце 'target' DataFrame target,
# что позволяет узнать распределение классов целевой переменной
target['target'].value_counts()

target
0    48888
1      961
Name: count, dtype: int64

### Проверка пропусков

In [57]:
# Выполняем анализ пропущенных значений в DataFrame purchases,
# используя словарь описаний полей fields_description для повышения наглядности результатов
analyze_na_values(purchases, fields_description)

,Поле,Тип данных,Количество пропусков,Доля пропусков (%),Описание


In [58]:
# Выполняем анализ пропусков в DataFrame messages,
# используя словарь описаний полей fields_description для более понятного и информативного отчёта
analyze_na_values(messages, fields_description)

,Поле,Тип данных,Количество пропусков,Доля пропусков (%),Описание


In [59]:
# Выполняем анализ пропущенных значений в DataFrame target,
# используя словарь описаний fields_description для улучшения информативности отчёта
analyze_na_values(target, fields_description)

,Поле,Тип данных,Количество пропусков,Доля пропусков (%),Описание


В данных отсутствуют пропуски.

### Проверка дубликатов в данных

In [60]:
# Вывод общего количества строк в DataFrame purchases,
# отражающего количество записей по покупкам клиентов по дням и товарам
print(f'Количество строк в данных о покупках клиентов по дням и по товарам: {len(purchases)}')

# Вывод количества полных дубликатов в DataFrame purchases,
# то есть строк, которые полностью совпадают по всем столбцам
print(f'Количество полных дубликатов в данных о покупках клиентов по дням и по товарам: {purchases.duplicated().sum()}')


Количество строк в данных о покупках клиентов по дням и по товарам: 202208
Количество полных дубликатов в данных о покупках клиентов по дням и по товарам: 73020


In [61]:
# Отбираем все строки в DataFrame purchases, которые являются полными дубликатами,
# включая все их вхождения (keep=False), и выводим первые 20 таких записей для анализа
purchases[purchases.duplicated(keep=False)].head(20)

,client_id,quantity,price,category_ids,purchase_date,message_id
11,1515915625491869271,2,599,"['4', '27', '350', '1392']",2022-05-16,1515915625491869271-2090-61a72488d6a0f
12,1515915625491869271,2,599,"['4', '27', '350', '1392']",2022-05-16,1515915625491869271-2090-61a72488d6a0f
21,1515915625566606509,1,299,"['5562', '5634', '5579', '710']",2022-05-16,1515915625566606509-4301-627b66245401d
22,1515915625566606509,1,299,"['5562', '5634', '5579', '710']",2022-05-16,1515915625566606509-4301-627b66245401d
41,1515915625468070175,1,2199,"['4', '27', '142', '496']",2022-05-17,1515915625468070175-4439-6283414668daa
42,1515915625468070175,1,2199,"['4', '27', '142', '496']",2022-05-17,1515915625468070175-4439-6283414668daa
58,1515915625468126078,1,1499,"['4', '28', '275', '421']",2022-05-17,1515915625468126078-4439-6283411f7c0cc
59,1515915625468126078,1,1499,"['4', '28', '275', '421']",2022-05-17,1515915625468126078-4439-6283411f7c0cc
61,1515915625468141317,1,49,"['4', '27', '38', '481']",2022-05-17,1515915625468141317-4439-6283412e01078
62,1515915625468141317,1,49,"['4', '27', '38', '481']",2022-05-17,1515915625468141317-4439-6283412e01078


В наборе данных о покупках клиентов по дням и товарам обнаружено 73 020 дублирующих записей из 202 208 — это составляет около 36%. Однако, такие дубли не обязательно являются ошибочными или техническими. Они могут отражать смысловые различия, связанные с деталями заказа, например:

- вариации характеристик товара в одном заказе, такие как цвет, размер или модификация;
- различные позиции, относящиеся к одной категории товаров;
- повторное приобретение идентичных товаров одним клиентом в течение дня.

Таким образом, часть дубликатов может являться валидными и отражать бизнес-реальность, а не ошибки в данных.

In [62]:
# Вывод общего количества строк в DataFrame messages,
# отражающего запись о рассылках, отправленных клиентам из таблицы покупок
print(f'Количество строк в данных о рассылках, которые были отправлены клиентам из таблицы покупок: {len(messages)}')

# Вывод количества полных дубликатов в DataFrame messages,
# то есть строк, которые полностью повторяются по всем столбцам
print(f'Количество полных дубликатов в данных о рассылках, которые были отправлены клиентам из таблицы покупок: {messages.duplicated().sum()}')


Количество строк в данных о рассылках, которые были отправлены клиентам из таблицы покупок: 12739798
Количество полных дубликатов в данных о рассылках, которые были отправлены клиентам из таблицы покупок: 48610


In [63]:
# Отбираем все строки из DataFrame messages, которые являются полными дубликатами,
# включая все экземпляры (keep=False), и показываем первые 20 таких записей для анализа
messages[messages.duplicated(keep=False)].head(20)


,bulk_campaign_id,client_id,message_id,event,channel,message_date,message_created_at
964231,5723,1515915625554535987,1515915625554535987-5723-62e2af08e00da,click,mobile_push,2022-07-28,2022-07-28 15:58:56
964232,5723,1515915625554535987,1515915625554535987-5723-62e2af08e00da,click,mobile_push,2022-07-28,2022-07-28 15:58:56
966465,5723,1515915625483569932,1515915625483569932-5723-62e2af0790ad5,click,mobile_push,2022-07-28,2022-07-28 16:07:12
966466,5723,1515915625483569932,1515915625483569932-5723-62e2af0790ad5,click,mobile_push,2022-07-28,2022-07-28 16:07:12
967281,5723,1515915625736038297,1515915625736038297-5723-62e2af0a17cba,click,mobile_push,2022-07-28,2022-07-28 16:09:56
967282,5723,1515915625736038297,1515915625736038297-5723-62e2af0a17cba,click,mobile_push,2022-07-28,2022-07-28 16:09:56
967566,5723,1515915625753898206,1515915625753898206-5723-62e2af0a21ecd,click,mobile_push,2022-07-28,2022-07-28 16:12:24
967567,5723,1515915625753898206,1515915625753898206-5723-62e2af0a21ecd,click,mobile_push,2022-07-28,2022-07-28 16:12:24
967903,5723,1515915625626349443,1515915625626349443-5723-62e2af09c0fd3,click,mobile_push,2022-07-28,2022-07-28 16:15:02
967904,5723,1515915625626349443,1515915625626349443-5723-62e2af09c0fd3,click,mobile_push,2022-07-28,2022-07-28 16:15:02


In [64]:
# Удаляем из DataFrame messages все полные дубликаты строк, оставляя только уникальные записи
messages = messages.drop_duplicates()

В данных о рассылках, направленных клиентам из таблицы покупок, обнаружено 48 610 полных дублирующих записей из общего количества 12 739 798, что составляет примерно 0,4%. Поскольку для этих дубликатов нет убедительных логических объяснений их присутствия в данных, они были удалены из набора для повышения качества анализа.


In [65]:
# Вывод общего количества строк в DataFrame target,
# отражающего записи о совершении клиентом покупки в целевом периоде
print(f'Количество строк в данных о совершении клиентом покупки в целевом периоде: {len(target)}')

# Вывод количества полных дубликатов в DataFrame target,
# то есть записей, которые полностью совпадают по всем столбцам
print(f'Количество полных дубликатов в данных о совершении клиентом покупки в целевом периоде: {target.duplicated().sum()}')


Количество строк в данных о совершении клиентом покупки в целевом периоде: 49849
Количество полных дубликатов в данных о совершении клиентом покупки в целевом периоде: 0


#### Проверка частичных дубликатов в данных

In [66]:
# Выполняем анализ количества уникальных значений в DataFrame purchases,
# используя словарь описаний полей fields_description для более понятной интерпретации результатов
analyze_unique_values(purchases, fields_description)


,Поле,Наименование поля,Уникальных значений,Доля уникальных (%),Самое частое значение,Кол-во самого частого значения,Доля самого частого значения (%)
5,message_id,идентификатор сообщения из рассылки,50204,25%,1515915625489095763-6251-6311b13a4cf78,365,0%
0,client_id,идентификатор клиента,49849,25%,1515915625853312319,346,0%
2,price,цена товара,3642,2%,999,10233,5%
3,category_ids,идентификаторы категорий,933,0%,"['4', '28', '57', '431']",8626,4%
4,purchase_date,дата покупки,642,0%,2022-11-11,5270,3%
1,quantity,количество единиц товара,16,0%,1,201323,100%


Во всех столбцах допустимо наличие значений дубликатов.

In [67]:
# Выполняем анализ уникальных значений в DataFrame messages,
# используя словарь descriptions для улучшения читаемости и пояснения результатов
analyze_unique_values(messages, fields_description)


,Поле,Наименование поля,Уникальных значений,Доля уникальных (%),Самое частое значение,Кол-во самого частого значения,Доля самого частого значения (%)
2,message_id,идентификатор сообщения из рассылки,9061667,71%,1515915625489095763-6251-6311b13a4cf78,1427,0%
6,message_created_at,дата создания рассылки,4103539,32%,2023-12-29 15:20:53,608,0%
1,client_id,идентификатор клиента,53329,0%,1515915625516327994,3088,0%
0,bulk_campaign_id,идентификатор рассылки,2709,0%,14272,104060,1%
5,message_date,дата действия рассылки,638,0%,2023-06-10,89173,1%
3,event,действие с сообщением,11,0%,send,9058174,71%
4,channel,канал рассылки,2,0%,mobile_push,7470472,59%


Во всех столбцах допустимо наличие значений дубликатов.

In [68]:
# Анализируем уникальные значения в DataFrame target,
# применяя словарь описаний fields_description для более понятной интерпретации результатов
analyze_unique_values(target, fields_description)


,Поле,Наименование поля,Уникальных значений,Доля уникальных (%),Самое частое значение,Кол-во самого частого значения,Доля самого частого значения (%)
0,client_id,идентификатор клиента,49849,100%,1515915625468060902,1,0%
1,target,клиент совершил покупку в целевом периоде (цел...,2,0%,0,48888,98%


Во всех столбцах кроме client_id (уникальный идентификатор клиента) допустимо наличие значений дубликатов.

**Выводы:**

**Итоговые наблюдения**:

- **Анализ типов данных**:
  - В данных о покупках клиентов по дням и товарам поле price было приведено из формата с плавающей точкой (float64) к целочисленному типу (int64), поскольку цена товара в этих данных не содержит дробной части.
 - **Проверка пропущенных значений**:
   - В данных о покупках клиентов отсутствуют пропуски.
   - Аналогично, в данных о рассылках клиентов пропуски не обнаружены.
 - **Анализ дублирующихся записей**:
   - **Полные дубликаты**:
    - В наборе данных о покупках выявлено 73 020 дублирующих записей из 202 208, что составляет примерно 36%. Однако эти дубликаты могут не быть ошибочными, а отражать семантические особенности данных, такие как:
      - различия в характеристиках товара в одном заказе, например, цвет, 
      - размер или модификация;
      - наличие нескольких товаров из одной категории;
      - повторные покупки одинаковых товаров в течение одного дня.
   - В данных о рассылках выявлено 48 610 полных дубликатов из 12 739 798 записей, то есть около 0,4%. Для этих дубликатов отсутствуют логичные объяснения, поэтому они были исключены из анализа.
 - **Частичные дубликаты**:
   - В данных о покупках присутствуют допустимые повторяющиеся значения в отдельных столбцах, что является нормальным.
   - В данных рассылок аналогично допускается наличие повторяющихся значений по отдельным признакам.
   
Таким образом, данные готовы к дальнейшему анализу с учётом специфики встречающихся дубликатов и отсутствия пропусков.

## Исследовательский анализ данных (до агрегации)

#### Данные о рассылках, которые были отправлены клиентам из таблицы покупок.

In [69]:
# Выводит подробную информацию о DataFrame messages, включая типы данных и количество непустых значений в каждом столбце
messages.info(show_counts=True)

<class 'pandas.core.frame.DataFrame'>
Index: 12691188 entries, 0 to 12739797
Data columns (total 7 columns):
 #   Column              Non-Null Count     Dtype 
---  ------              --------------     ----- 
 0   bulk_campaign_id    12691188 non-null  int64 
 1   client_id           12691188 non-null  int64 
 2   message_id          12691188 non-null  object
 3   event               12691188 non-null  object
 4   channel             12691188 non-null  object
 5   message_date        12691188 non-null  object
 6   message_created_at  12691188 non-null  object
dtypes: int64(2), object(5)
memory usage: 774.6+ MB


In [ ]:
# Подсчитываем количество вхождений каждого уникального значения в столбце 'message_id' DataFrame messages,
# что позволяет оценить распределение идентификаторов сообщений
messages['message_id'].value_counts()


In [ ]:
# Список категориальных признаков в DataFrame messages,
# которые будут использоваться для анализа или визуализации
messages_categorial_features = [
    'client_id', 
    'bulk_campaign_id', 
    'event', 
    'channel'
]

In [ ]:
# Визуализируем распределение категориальных признаков из списка messages_categorial_features в DataFrame messages,
# используя идентификатор 'message_id' для подсчётов,
# показываем топ 15 самых частых значений по каждому признаку,
# сортируем значения по убыванию,
# задаём размер фигуры и максимальное количество графиков на одном рисунке,
# применяем словарь описаний fields_description и ограничиваем длину подписей по оси X до 11 символов
plot_categorical_features(
    messages,
    categorical_cols=messages_categorial_features,
    id_col='message_id',
    top_n=15,
    sort_desc=True,
    figsize=(12, 6),
    max_plots_per_figure=6,
    descriptions=fields_description,
    label_wrap_length=11
)


In [ ]:
# Список временных признаков в DataFrame messages,
# которые планируется использовать для анализа распределения событий по времени
messages_date_features = [
    'message_date', 
    # 'message_created_at'  # временной признак пока не используется
]


In [ ]:
# Сохраняем общее количество строк в DataFrame messages
messages_rows_count = messages.shape[0]

# Считаем количество строк, у которых даты в столбцах 'message_date' и 'message_created_at' совпадают по дню (игнорируем время)
equal_dates_count = len(messages[
    pd.to_datetime(messages['message_date']).dt.date == pd.to_datetime(messages['message_created_at']).dt.date
])

# Выводим количество строк, где дата в 'message_date' и 'message_created_at' отличается
print(f'Дата в "message_date" и "message_created_at" не совпадает в {messages_rows_count - equal_dates_count} строках.')


In [ ]:
# Визуализируем распределение событий по датам из списка messages_date_features в DataFrame messages,
# используя 'message_id' как идентификатор, настраиваем размер фигур, количество графиков на фигуру,
# формат отображения дат и описания столбцов из словаря fields_description.
plot_date_distribution(
    messages,
    date_cols=messages_date_features,
    id_col='message_id',
    figsize=(12, 6),
    max_plots_per_figure=6,
    date_format='%Y-%m-%d',
    descriptions=fields_description
)


In [ ]:
# [jg:person_53] визуализация распределения событий из колонок messages_date_features DataFrame messages,
# агрегирование по неделям (freq='W'), использование 'message_id' как идентификатора,
# настройка размера фигуры, максимального количества графиков на одной фигуре,
# форматирование дат и отображение описаний столбцов из словаря fields_description
plot_date_distribution(
    messages,
    date_cols=messages_date_features,
    id_col='message_id',
    figsize=(14, 7),
    max_plots_per_figure=6,
    date_format='%Y-%m-%d',
    freq='W',
    descriptions=fields_description
)


#### Данные о покупках клиентов по дням и по товарам.

In [ ]:
# Выводит сводную информацию о DataFrame purchases: количество непустых значений в каждом столбце,
# типы данных и общий объём занимаемой памяти
purchases.info()


In [ ]:
# Подсчитываем количество уникальных значений и частоту их появления в столбце 'message_id' DataFrame purchases,
# что помогает оценить распределение идентификаторов сообщений
purchases['message_id'].value_counts()


In [ ]:
# Список категориальных признаков в DataFrame purchases,
# выбранных для последующего анализа или визуализации
purchases_categorial_features = [
    'client_id',
    'category_ids'
]


In [ ]:
# Визуализация распределения категориальных признаков из списка purchases_categorial_features в DataFrame purchases
# Используется 'message_id' как идентификатор для подсчёта частоты,
# отображаются топ-15 наиболее частых значений для каждого признака,
# сортировка по убыванию, задан размер фигуры и максимальное количество графиков на одной фигуре,
# используется словарь описаний полей fields_description для информативных заголовков,
# подписи меток по оси X обрезаются до 30 символов с поворотом на 25 градусов для читаемости
plot_categorical_features(
    purchases,
    categorical_cols=purchases_categorial_features,
    id_col='message_id',
    top_n=15,
    sort_desc=True,
    figsize=(12, 6),
    max_plots_per_figure=6,
    descriptions=fields_description,
    label_wrap_length=30,
    label_rotation=25
)


In [ ]:
# Список числовых признаков в DataFrame purchases,
# которые будут использоваться для анализа или моделирования
purchases_numeric_features = [
    'quantity',
    'price'
]


In [ ]:
# Строим комбинированные графики (гистограмма и боксплот) для числовых признаков из списка purchases_numeric_features
# по данным в DataFrame purchases с использованием описаний полей из словаря fields_description
draw_combined_hist_boxplot(purchases, fields_description, fields=purchases_numeric_features)


In [ ]:
# Список временных признаков в DataFrame purchases,
# предназначенных для анализа и визуализации распределения по времени
purchases_date_features = [
    'purchase_date'
]


In [ ]:
# Визуализация распределения событий по датам из столбцов purchases_date_features DataFrame purchases,
# использование 'message_id' как идентификатора,
# с настройкой размера фигуры, максимального числа графиков на одном рисунке,
# формата отображения дат и описаний столбцов из словаря fields_description
plot_date_distribution(
    purchases,
    date_cols=purchases_date_features,
    id_col='message_id',
    figsize=(14, 7),
    max_plots_per_figure=6,
    date_format='%Y-%m-%d',
    descriptions=fields_description
)


In [ ]:
# Визуализация распределения событий по датам из столбцов purchases_date_features DataFrame purchases,
# с агрегированием данных по неделям (freq='W'),
# использование 'message_id' как идентификатора,
# настройка размера графика, количества графиков на одной фигуре,
# отображение дат в формате 'ГГГГ-ММ-ДД', а также использование описаний из fields_description
plot_date_distribution(
    purchases,
    date_cols=purchases_date_features,
    id_col='message_id',
    figsize=(14, 7),
    max_plots_per_figure=6,
    date_format='%Y-%m-%d',
    freq='W',
    descriptions=fields_description
)


**Выводы:**

 - **Данные по рассылкам, направленным клиентам из таблицы покупок:**

   - Поле 'message_id' содержит хэш-значения, при этом в наборе данных встречаются повторы одних и тех же идентификаторов.

   - Поле 'client_id' представляет собой числовой идентификатор клиента, и в данных присутствуют повторяющиеся значения этого идентификатора.

   - Поле 'bulk_campaign_id' — числовой идентификатор рассылочной кампании, в рамках которой осуществляется массовая отправка сообщений.

   - Значение поля 'event' характеризует тип действия с сообщением; три наиболее распространённых события — «send» (отправка, 71,4%), «open» (открытие, 24,1%) и «click» (клик, 3,8%), вместе они составляют 99% всех зафиксированных событий.

   - Поле 'channel' определяет канал рассылки: «mobile push» (58,9%) и «email» (41,1%).

   - Пара полей 'message_date' и 'message_created_at' показывает дату и дату с временем отправки сообщения соответственно. Анализ временных рядов показывает явный рост среднего количества отправленных сообщений как в дневном, так и в недельном разрезе, начиная со второго полугодия 2023 года.

 - **Данные о покупках клиентов по дням и товарам:**

   - Поле 'client_id' — числовой идентификатор клиента, с повторяющимися значениями в данных.

   - Поле 'category_ids' содержит список идентификаторов категорий товара, который иногда может быть пустым. Из-за изменений в структуре категорий будут использоваться последние (листовые) уровни категорий, соответствующие, скорее всего, товарным кодам (PLU) или близким по смыслу категориям.

   - Поле 'quantity' отражает количество приобретённых единиц товара — среднее и медианное значения равны 1, что говорит о том, что большинство покупок — по одному товару.

   - Поле 'price' содержит цену приобретаемого товара. Средняя цена превышает медиану (1199 против 987), что свидетельствует о присутствии выбросов — товаров с существенно высокой ценой.

   - Поле 'purchase_date' фиксирует дату покупки. Анализ по дням и неделям указывает на заметное снижение среднего количества покупок как в дневном, так и в недельном разрезе, начиная со второго полугодия 2023 года.

Таким образом, данные показывают важные особенности рассылок и покупательской активности, которые следует учитывать при дальнейшем анализе и моделировании.

## Объединение данных

In [ ]:
# Объединяем DataFrame messages и purchases по колонкам 'client_id' и 'message_id' с использованием внешнего соединения (outer),
# что сохраняет все записи из обоих наборов данных, а параметр validate='many_to_many' гарантирует, что соединение допускает множественные совпадения с обеих сторон
data = pd.merge(messages, purchases, on=['client_id', 'message_id'], how='outer', validate='many_to_many')


In [ ]:
# Объединяем DataFrame target с ранее сформированным DataFrame data по колонке 'client_id' с левым соединением (left join),
# что сохраняет все записи из target и добавляет соответствующие данные из data,
# параметр validate='one_to_many' проверяет, что для каждого 'client_id' в target может быть несколько строк в data
data = pd.merge(target, data, on='client_id', how='left', validate='one_to_many')


## Агрегация и создание новых признаков

#### Создание вспомогательных функций

In [ ]:
# Функция пытается преобразовать строковое представление списка категорий в объект Python (список),
# если входное значение x является строкой, используя literal_eval для безопасного разбора,
# если x уже список, возвращает его без изменений,
# в случае ошибок или неподходящего типа возвращает пустой список,
def parse_categories(x):
    try:
        if isinstance(x, str):
            return literal_eval(x)
        return x if isinstance(x, list) else []
    except:
        return []


In [ ]:
# Функция возвращает последний элемент из списка категорий cat_list,
# если входной параметр является непустым списком,
# в противном случае возвращает None
def get_last_category(cat_list):
    if isinstance(cat_list, list) and len(cat_list) > 0:
        return str(cat_list[-1])
    return None


In [ ]:
# Выводит сводную информацию о DataFrame data, включая типы данных и количество не пропущенных значений по каждому столбцу
data.info(show_counts=True)


#### Инициализация и подготовка данных

In [ ]:
# Преобразуем столбцы с датами в формат datetime для корректной работы с временными данными
data['message_date'] = pd.to_datetime(data['message_date'])
data['message_created_at'] = pd.to_datetime(data['message_created_at'])
data['purchase_date'] = pd.to_datetime(data['purchase_date'])

# Заполняем пропуски в количестве товара и цене нулями, так как отсутствие значения подразумевает отсутствие покупки
data['quantity'] = data['quantity'].fillna(0)
data['price'] = data['price'].fillna(0)
# Заполняем пропуски в категориях пустыми списками в строковом формате
data['category_ids'] = data['category_ids'].fillna('[]')

# Преобразуем строковые представления списков категорий в реальные списки с помощью функции parse_categories
data['category_list'] = data['category_ids'].apply(parse_categories)
# Извлекаем последний уровень категории (листовой уровень) с помощью функции get_last_category, который зачастую соответствует коду товара (PLU)
data['plu'] = data['category_list'].apply(get_last_category)


#### Агрегация данных по клиентам

In [ ]:
# Выполняем агрегирование данных по каждому клиенту (client_id):
# - для поля 'target' берём первое значение (предположительно целевой признак)
# - для поля 'bulk_campaign_id' считаем количество уникальных рассылок, связанных с клиентом
aggr_data = data.groupby('client_id').agg({
    'target': 'first',
    'bulk_campaign_id': 'nunique'
}).reset_index()

# Переименовываем столбцы для удобства восприятия:
# 'unique_campaigns' — количество уникальных кампаний рассылок на клиента
aggr_data.columns = ['client_id', 'target', 'unique_campaigns']



#### Статистика по каналам

In [ ]:
# Группируем данные по клиенту ('client_id') и каналу рассылки ('channel'),
# затем считаем количество записей в каждой группе,
# преобразуем результат в таблицу с разделением по каналам (unstack),
# заполняя отсутствующие значения нулями
channel_counts = data.groupby(['client_id', 'channel']).size().unstack(fill_value=0)

# Переименовываем столбцы для ясности, добавляя префикс 'channel_' перед названием канала
channel_counts.columns = [f'channel_{col}' for col in channel_counts.columns]


#### Статистика по покупкам

In [ ]:
# Отбираем только те записи из data, где количество купленных товаров больше нуля,
# затем группируем данные по клиентам и дате покупки,
# суммируя количество товаров и стоимость за каждый день для каждого клиента
daily_purchase = data[data['quantity'] > 0].groupby(['client_id', 'purchase_date']).agg({
    'quantity': 'sum',
    'price': 'sum'
}).reset_index()

# Для каждого клиента агрегируем ежедневные данные:
# - среднее значение покупки (средняя сумма потраченных средств)
# - дату первой покупки
# - дату последней покупки
# - количество уникальных дней с покупками
purchase_stats = daily_purchase.groupby('client_id').agg({
    'price': ['mean'],
    'purchase_date': ['min', 'max', 'nunique']
}).reset_index()

# Переименовываем столбцы для удобства восприятия
purchase_stats.columns = ['client_id', 'avg_purchase_sum',
                          'first_purchase', 'last_purchase', 'unique_purchase_days']

# Вычисляем длительность периода совершения покупок в днях
purchase_stats['purchase_period_days'] = (purchase_stats['last_purchase'] - purchase_stats['first_purchase']).dt.days

# Удаляем колонки с датами первой и последней покупки,
# так как они заменены на длительность периода
purchase_stats = purchase_stats.drop(['first_purchase',  'last_purchase'], axis=1)



#### Статистика по кодами товаров

In [ ]:
# Считаем для каждого клиента (client_id):
# - количество уникальных продуктов (plu)
# - общее число покупок продуктов (с учётом повторов)
plu_stats = data.groupby('client_id').agg({
    'plu': ['nunique', 'count'],
}).reset_index()
plu_stats.columns = ['client_id', 'unique_plu_count', 'total_plu_purchases']

# Подсчитываем суммарное количество по каждому продукту (plu) среди всех клиентов
plu_popularity = data.groupby('plu')['quantity'].sum().reset_index()
# Сортируем продукты по количеству в порядке убывания популярности
plu_popularity = plu_popularity.sort_values('quantity', ascending=False)
# Выбираем топ-20 самых популярных продуктов
top_20_plu = plu_popularity.head(20)['plu'].tolist()

# Отбираем данные только по топ-20 популярным продуктам
top_plu_data = data[data['plu'].isin(top_20_plu)]
# Для каждого клиента считаем количество уникальных продуктов из топ-20 в его покупках
client_top_plu = top_plu_data.groupby('client_id').agg({
    'plu': ['nunique'],
}).reset_index()
client_top_plu.columns = ['client_id', 'unique_top20_plu']

# Объединяем общую статистику plu_stats с информацией о топ-20 продуктах по client_id
plu_stats = plu_stats.merge(client_top_plu, on='client_id', how='left')
# Заполняем пропущенные значения нулями (например, если клиент не покупал топ-20 продуктов)
plu_stats = plu_stats.fillna(0)


#### Статистика по самому популярному коду товара

In [ ]:
# Подсчитываем количество покупок каждого продукта (plu) каждым клиентом
client_plu_counts = data.groupby(['client_id', 'plu']).size().reset_index(name='count')

# Для каждого клиента находим индекс продукта, который покупался чаще всего
idx = client_plu_counts.groupby('client_id')['count'].idxmax()

# Извлекаем наиболее частые покупки каждого клиента с их количеством
top_client_plu = client_plu_counts.loc[idx][['client_id', 'plu', 'count']]

# Переименовываем столбцы для ясности
top_client_plu.columns = ['client_id', 'most_frequent_plu', 'most_frequent_plu_count']

# Объединяем полученную информацию с основным DataFrame plu_stats по client_id
plu_stats = plu_stats.merge(top_client_plu, on='client_id', how='left')

# Вычисляем долю самой часто приобретаемой позиции из общего количества покупок по продуктам для каждого клиента,
# при условии, что общее количество покупок больше нуля, иначе ставим ноль
plu_stats['most_frequent_plu_ratio'] = plu_stats.apply(
    lambda x: x['most_frequent_plu_count'] / x['total_plu_purchases'] if x['total_plu_purchases'] > 0 else 0,
    axis=1
)

# Удаляем столбец 'total_plu_purchases', так как он используется для расчёта и далее не требуется
plu_stats = plu_stats.drop('total_plu_purchases', axis=1)


#### Объединение созданных признаков

In [ ]:
# Создаём копию агрегированных данных клиентов для дальнейшей обработки
final_data = aggr_data.copy()

# Объединяем с данными по каналам рассылки по 'client_id' с левым соединением,
# чтобы добавить информацию о количестве рассылок по разным каналам для каждого клиента
final_data = pd.merge(final_data, channel_counts, on='client_id', how='left')

# Объединяем с агрегированными статистиками по покупкам клиентов
final_data = pd.merge(final_data, purchase_stats, on='client_id', how='left')

# Объединяем с агрегированными статистиками по продуктовым категориям
final_data = pd.merge(final_data, plu_stats, on='client_id', how='left')

# Список столбцов с показателями покупок, в которых пропущенные значения заменяются на нули
purchase_columns = [
    'avg_purchase_sum', 'unique_purchase_days', 'purchase_period_days'
]

# Заполняем пропуски в перечисленных столбцах нулями для корректной работы моделей и анализа
for col in purchase_columns:
    final_data[col] = final_data[col].fillna(0)


## Исследовательский анализ данных (после агрегации)

In [ ]:
# Выводит сводную информацию о DataFrame final_data,
# включая количество непустых значений в каждом столбце, типы данных и использование памяти
final_data.info()

In [ ]:
# Словарь с описаниями итоговых признаков из объединённого датасета final_data,
# позволяющий удобно интерпретировать каждый столбец при анализе и построении отчетов
final_fields_description = {
    'client_id': 'идентификатор клиента',
    'target': 'клиент совершил покупку в целевом периоде (целевая переменная)',

    'unique_campaigns': 'количество уникальных маркетинговых кампаний, в которые был включен клиент',

    'channel_email': 'количество взаимодействий с клиентом через электронную почту',
    'channel_mobile_push': 'количество взаимодействий с клиентом через push-уведомления на мобильное устройство',

    'avg_purchase_sum': 'средняя стоимость одной покупки (средний чек)',

    'unique_purchase_days': 'количество уникальных дней, в которые клиент совершал покупки',
    'purchase_period_days': 'количество дней между первой и последней покупкой клиента (длина клиентской истории)',

    'unique_plu_count': 'количество уникальных товарных позиций (PLU), которые покупал клиент',

    'most_frequent_plu': 'идентификатор товарной позиции (PLU), которую клиент покупал чаще всего',
    'most_frequent_plu_count': 'количество покупок самого частого товара у клиента',
    'most_frequent_plu_ratio': 'доля покупок самого частого товара от общего числа покупок клиента',

    'unique_top20_plu': 'количество уникальных товаров из общего ТОП-20 популярных товаров, которые покупал клиент',
}


In [ ]:
# Подсчитываем количество записей для каждого уникального значения 'client_id' в DataFrame final_data,
# что позволяет понять распределение активности или частоту появления клиентов в данных
final_data['client_id'].value_counts()


In [ ]:
# Список категориальных признаков в DataFrame data,
# включающий наиболее часто покупаемые товарные позиции (PLU)
data_categorial_features = [
    'most_frequent_plu'
]

In [ ]:
# Построение распределения категориальных признаков из списка data_categorial_features в DataFrame final_data,
# используя 'client_id' как идентификатор для подсчётов,
# отображаем топ-20 наиболее частых значений каждого признака,
# сортируем по убыванию, настраиваем размер фигуры и количество графиков на ней,
# применяем описания из словаря final_fields_description,
# а также ограничиваем длину подписей по оси X до 11 символов для удобства чтения.
plot_categorical_features(
    final_data,
    categorical_cols=data_categorial_features,
    id_col='client_id',
    top_n=20,
    sort_desc=True,
    figsize=(12, 6),
    max_plots_per_figure=6,
    descriptions=final_fields_description,
    label_wrap_length=11
)


In [ ]:
# Список числовых признаков в DataFrame data,
# предназначенных для использования в анализе и построении моделей
data_numeric_features = [
    'unique_campaigns',         # количество уникальных маркетинговых кампаний, охватывающих клиента
    'channel_email',            # число взаимодействий с клиентом через email-канал
    'channel_mobile_push',      # число взаимодействий с клиентом через push-уведомления
    'avg_purchase_sum',         # средняя сумма покупки (средний чек)
    'unique_purchase_days',     # количество уникальных дней с покупками
    'purchase_period_days',     # длительность клиентской истории в днях
    'unique_plu_count',         # число уникальных товарных позиций, которые покупал клиент
    'unique_top20_plu',         # количество уникальных товаров из топ-20 популярных покупок клиента
    'most_frequent_plu_count',  # количество покупок самого частого товара
    'most_frequent_plu_ratio'   # доля покупок самого частого товара от всех покупок клиента
]


In [ ]:
# Строим комбинированные графики (гистограммы и боксплоты) для числовых признаков из списка data_numeric_features,
# используя данные из DataFrame final_data и применяя описания признаков из словаря final_fields_description
draw_combined_hist_boxplot(final_data, final_fields_description, fields=data_numeric_features)

**Выводы:**

Поле 'client_id' выступает идентификатором клиента и служит первичным ключом в подготовленных данных для машинного обучения.

Поле 'most_frequent_plu' указывает на товарную позицию (PLU), которую клиент приобретал наиболее часто. При этом топ-20 самых популярных товаров составляет примерно 44% от всех продаж.

Поле 'unique_campaigns' отражает количество уникальных маркетинговых кампаний, в которых участвовал клиент. Среднее значение составляет около 157 кампаний, медиана — 150, что говорит о наличии аномально активных клиентов, получавших уведомления из значительно большего количества кампаний.

Поля 'channel_email' и 'channel_mobile_push' показывают количество взаимодействий с клиентом по электронной почте и через мобильные push-уведомления соответственно. Медианы примерно равны — 129 для email и 125 для push, при этом средние значения выше медиан, что указывает на существование клиентов с особенно высоким уровнем взаимодействия.

Поле 'avg_purchase_sum' — средняя сумма одной покупки. Среднее значение (~16.012) существенно выше медианного (~8.396), что свидетельствует о значительном числе покупок с высокими затратами, влияющих на распределение.

Поля 'unique_purchase_days' и 'purchase_period_days' характеризуют количество дней, в которые клиент совершал покупки, и длительность его покупательской активности соответственно. Медианы равны 1 и 0, что говорит о том, что большинство клиентов делают покупки лишь один раз или практически не повторяют покупки.

Поле 'unique_plu_count' — число уникальных товарных позиций, приобретённых клиентом. Медианное значение равно 1, что говорит о том, что покупатели обычно приобретают товары из одной категории, а среднее (~1.83) выше медианы, что указывает на наличие клиентов с широким ассортиментом покупок.

Поле 'unique_top20_plu' — количество уникальных товаров из 20 самых популярных, приобретённых клиентом. Медианное число равно 1, среднее — 0.71, что подтверждает, что покупатели приобретают в основном один товар из топ-20.

Поле 'most_frequent_plu_count' отражает число покупок самого часто приобретаемого товара. Медианное значение — 6, среднее — 9.84, что говорит о клиентах с большим количеством повторных покупок одного продукта.

Поле 'most_frequent_plu_ratio' — доля покупок самого популярного товара относительно всех покупок клиента. Медиана на уровне 100%, среднее около 80%, поддерживая вывод о том, что часто клиент совершает все или большую часть покупок одного товара.

Таким образом, анализ данных указывает на преобладание покупателей, ориентированных на ограниченный ассортимент товаров, с единичными покупками и различной степенью вовлечённости в маркетинговые кампании.

## Корреляционный анализ

### Корреляция основаная на хи-квадрат статистике с нормализацией

In [ ]:
# Список числовых признаков, используемых для анализа и построения моделей,
# включающий показатели маркетинговых кампаний, канальных взаимодействий, стоимости и покупательской активности
numerical_features = [
    'unique_campaigns',         # Количество различных маркетинговых кампаний, затронувших клиента
    'channel_email',            # Число взаимодействий через электронную почту
    'channel_mobile_push',      # Число взаимодействий через push-уведомления на мобильных устройствах
    'avg_purchase_sum',         # Средняя сумма покупки (средний чек)
    'unique_purchase_days',     # Количество уникальных дней с покупками
    'purchase_period_days',     # Длительность истории покупок клиента в днях
    'most_frequent_plu_count',  # Число покупок самого часто приобретаемого товара
    'most_frequent_plu_ratio',  # Доля покупок наиболее частого товара от общего числа покупок
    'unique_plu_count',         # Количество уникальных товарных позиций, приобретённых клиентом
    'unique_top20_plu',         # Количество уникальных товаров из ТОП-20, которые покупал клиент
]


In [ ]:
# Список категориальных признаков, включающий идентификатор наиболее часто покупаемой товарной позиции (PLU)
categorial_futures = [
    'most_frequent_plu'
]

In [ ]:
# Объединяем списки числовых и категориальных признаков в один список для анализа корреляций
correlation_features = numerical_features + categorial_futures

# Добавляем целевую переменную 'target' в список признаков для включения в анализ
correlation_features.append('target')


In [ ]:
# Вычисляем расширенную корреляционную матрицу Phik для признаков в списке correlation_features,
# используя список числовых признаков numerical_features для корректной обработки непрерывных данных
correlation_matrix = final_data[correlation_features].phik_matrix(numerical_features)


In [ ]:
# Строим тепловую карту (heatmap) на основе корреляционной матрицы correlation_matrix,
# используя описания признаков из словаря final_fields_description для подписей,
# и задаём размер ячейки равным 1 для удобного отображения
draw_heatmap_by_grouped_data(correlation_matrix, final_fields_description, cell_size=1)


In [ ]:
# Получаем пары признаков из корреляционной матрицы correlation_matrix,
# у которых абсолютное значение коэффициента корреляции превышает порог 0.6
get_high_correlations(correlation_matrix, threshold=0.6)


**Анализ корреляций с целевой переменной «Совершение покупки клиентом в целевом периоде»:**

Не обнаружено признаков с выраженной линейной зависимостью (корреляция ≥ 0.3) с целевой переменной. Максимальное значение корреляции отмечено у признака «идентификатор товарной позиции (PLU), которую клиент покупал наиболее часто» и составило всего 0.13. Это указывает на то, что явная линейная связь между отдельными входными признаками и приобретением товара в целевом периоде довольно слабая. Вероятно, для успешного прогнозирования потребуется применение более сложных моделей, способных улавливать нелинейные зависимости.

**Корреляции между признаками модели:**

Сильные взаимосвязи (корреляция ≥ 0.6) обнаружены между следующими парами признаков:

- channel_email и most_frequent_plu_count — 0.89
- channel_email и unique_purchase_days — 0.87
- most_frequent_plu_ratio и unique_plu_count — 0.75
- channel_mobile_push и most_frequent_plu_count — 0.72
- unique_campaigns и channel_mobile_push — 0.71
- unique_purchase_days и most_frequent_plu_count — 0.68
- channel_mobile_push и most_frequent_plu — 0.67
- unique_campaigns и most_frequent_plu — 0.66
- unique_top20_plu и most_frequent_plu — 0.65
- most_frequent_plu_count и unique_plu_count — 0.63
- unique_plu_count и unique_top20_plu — 0.63

Эти данные показывают наличие тесных связей между маркетинговыми взаимодействиями и разнообразием покупок клиентов, что следует учитывать при построении моделей и интерпретации результатов.

**Выводы:**

Отсутствие ярко выраженных прямых линейных связей с целевой переменной говорит о том, что между признаками и вероятностью покупки в заданный период существуют сложные и нелинейные отношения.

Кроме того, исследуемые признаки, созданные на основе исходных данных, демонстрируют значимые взаимосвязи между собой, что отражает их взаимозависимый и многогранный характер.


## Обучение модели

### Инициализация и подготовка данных

In [ ]:
# Список числовых признаков, используемых для анализа и построения модели
num_columns = [
    'unique_campaigns', 
    'channel_email',
    'channel_mobile_push', 
    'avg_purchase_sum', 
    'unique_purchase_days',
    'purchase_period_days',
    'most_frequent_plu_count',
    'most_frequent_plu_ratio',
    'unique_plu_count',
    'unique_top20_plu',
    ]


In [ ]:
# Список категориальных признаков, используемых для анализа и построения модели
cat_columns = [
    'most_frequent_plu'
]


### Создание пайплайнов предобработки данных и обучения

In [ ]:
# Конвейер обработки числовых признаков: заполнение пропущенных значений медианой и масштабирование в диапазон [0,1]
num_pipe = Pipeline([
    ('simpleImputer_num', SimpleImputer(missing_values=np.nan, strategy='median')),
    ('scaler', MinMaxScaler())
])


In [ ]:
# Конвейер обработки категориальных признаков: заполнение пропущенных значений наиболее частыми и one-hot кодирование
cat_pipe = Pipeline([
    ('simpleImputer_cat', SimpleImputer(missing_values=np.nan, strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


In [ ]:
# Комбинированный препроцессор данных: применение конвейеров для категориальных и числовых признаков, остальные признаки пропускаются без изменений
data_preprocessor = ColumnTransformer([
     ('cat', cat_pipe, cat_columns),
     ('num', num_pipe, num_columns),
], remainder='passthrough')


In [ ]:
# Общий конвейер: предварительная обработка данных с последующим этапом модели (пока модель не задана)
pipeline = Pipeline([
    ('preprocessor', data_preprocessor),
    ('models', None)
])


### Определение гиперпараметров для обучения моделей

In [ ]:
# Расчет веса положительного класса для компенсации дисбаланса: отношение числа отрицательных примеров к положительным
SCALE_POS_WEIGHT = len(final_data[final_data['target'] == 0]) / len(final_data[final_data['target'] == 1])


In [ ]:
# Сетка гиперпараметров для перебора в GridSearchCV/RandomizedSearchCV с несколькими моделями и их настройками
param_grid = [
    {
        'models': [LogisticRegression(
            random_state=RANDOM_STATE,
            max_iter=1000,
            class_weight='balanced'
        )],
        'models__C': [0.1, 1.0, 10.0],
        'models__penalty': ['l1', 'l2'],
        'models__solver': ['liblinear'],
        'models__tol': [1e-4, 1e-3],
        'models__intercept_scaling': [1, 2],
    },
    {
        'models': [XGBClassifier(
            random_state=RANDOM_STATE,
            eval_metric='logloss',
            scale_pos_weight=SCALE_POS_WEIGHT
        )],
        'models__n_estimators': [200, 300],
        'models__max_depth': [3, 5],
        'models__learning_rate': [0.1, 0.2],
        'models__subsample': [0.8],
    },
    {
        'models': [LGBMClassifier(
            random_state=RANDOM_STATE,
            verbose=-1,
            scale_pos_weight=SCALE_POS_WEIGHT
        )],
        'models__n_estimators': [200, 300],
        'models__max_depth': [3, 5],
        'models__learning_rate': [0.1, 0.2]
    },
    {
        'models': [CatBoostClassifier(
            random_state=RANDOM_STATE, 
            verbose=False,
            allow_writing_files=False,
            scale_pos_weight=SCALE_POS_WEIGHT
        )],
        'models__iterations': [200, 300],
        'models__depth': [3, 5],
        'models__learning_rate': [0.05, 0.1],
        'models__l2_leaf_reg': [1, 3]
    }
]


### Кросс-валидация моделей

In [ ]:
# Разделение данных на признаки (X) и целевую переменную (y), исключая идентификатор клиента и целевой столбец из признаков
X = final_data.drop(['client_id', 'target'], axis=1)
y = final_data['target']


In [ ]:
# Разделение данных на обучающую и тестовую выборки с сохранением пропорций классов (стратификация) и фиксированным random_state для воспроизводимости
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
    shuffle=True
)


In [ ]:
# Настройка GridSearchCV для перебора гиперпараметров моделей внутри конвейера с 5-кратной кросс-валидацией и метрикой ROC AUC
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=3
)


In [ ]:
# Обучение модели с подбором гиперпараметров по сетке на всех данных с использованием GridSearchCV
grid.fit(X, y)


### Анализ результатов и выбор лучшей модели машинного обучения

In [ ]:
# Создание датафрейма с результатами кросс-валидации и вывод 10 лучших моделей по рангу качества
results = pd.DataFrame(grid.cv_results_)
print('Топ-10 лучших моделей машинного обучения:')
results.sort_values('rank_test_score')[:10]


In [ ]:
# Выбор лучшей модели, найденной в процессе GridSearchCV, и вывод её параметров
model = grid.best_estimator_
print("Лучшая модель машинного обучения: \n\n", model)


In [ ]:
# Вывод лучшего значения метрики ROC-AUC, достигнутого выбранной моделью в ходе кросс-валидации
print('Метрика ROC-AUC для лучшей модели машинного обучения:\n', grid.best_score_)


### Оценка модели машинного обучения на тестовой выборке

In [ ]:
# Предсказание целевой переменной на тестовой выборке с помощью лучшей обученной модели
y_pred = model.predict(X_test)


In [ ]:
# Получение вероятностей принадлежности к положительному классу на тестовой выборке с помощью лучшей модели
y_proba = model.predict_proba(X_test)[:, 1]


In [ ]:
# Вычисление значений ложноположительной (FPR) и истинноположительной (TPR) частей ROC-кривой и порогов классификации
fpr, tpr, threshold = roc_curve(y_test, y_proba)


In [ ]:
# Вывод основных метрик качества лучшей модели на тестовой выборке: Precision, Recall, F2-score и ROC-AUC
print("Основные метрики лучшей модели машинного обучения на тестовой выборке:")
print(f"Precision: {precision_score(y_test, y_pred):.2f}")
print(f"Recall: {recall_score(y_test, y_pred):.2f}")
print(f"F2-score = {fbeta_score(y_test, y_pred, beta=2):.2f}")
print(f"ROC-AUC = {roc_auc_score(y_test, y_proba):.2f}")


In [ ]:
# Построение ROC-кривой: график зависимости ложноположительной ошибки (FPR) от полноты (TPR)
plt.plot(fpr, tpr)
plt.title("График зависимости FPR от TPR")
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.show()


In [ ]:
# Обучение и предсказание базовой модели DummyClassifier со стратегией stratified для сравнения с основной моделью
dummy = DummyClassifier(strategy='stratified', random_state=RANDOM_STATE)
dummy.fit(X_train, y_train)

y_pred_dummy = dummy.predict(X_test)
y_proba_dummy = dummy.predict_proba(X_test)[:, 1]
fpr_dummy, tpr_dummy, threshold_dummy = roc_curve(y_test, y_proba)


In [ ]:
# Вывод основных метрик DummyClassifier и сравнение ROC-AUC базовой модели с лучшей моделью на тестовой выборке
print("Метрики DummyClassifier на тестовой выборке:")
print(f"Precision: {precision_score(y_test, y_pred_dummy):.2f}")
print(f"Recall: {recall_score(y_test, y_pred_dummy):.2f}")
print(f"F2-score = {fbeta_score(y_test, y_pred_dummy, beta=2):.2f}")
print(f"ROC-AUC = {roc_auc_score(y_test, y_proba_dummy):.2f}")

print("\nСравнение с лучшей моделью:")
print(f"ROC-AUC лучшей модели: {roc_auc_score(y_test, y_proba):.4f}")
print(f"ROC-AUC DummyClassifier: {roc_auc_score(y_test, y_proba_dummy):.4f}")


In [ ]:
# Построение ROC-кривой для базовой модели DummyClassifier: зависимость ложноположительной ошибки (FPR) от полноты (TPR)
plt.plot(fpr_dummy, tpr_dummy)
plt.title("График зависимости FPR от TPR")
plt.xlabel('False Positive Rate (FPR)')
plt.ylabel('True Positive Rate (TPR)')
plt.show()


### Подбор порога классификации

In [ ]:
# Поиск оптимального порога классификации по метрикам Precision, Recall, F1, F2 и количеству предсказанных положительных классов
thresholds = np.arange(0.1, 0.9, 0.01)
results = []

print("Поиск оптимального порога классификации")
print("=" * 75)
print(f"{'Порог':<8} | {'Precision':<10} | {'Recall':<8} | {'F1':<8} | {'F2':<8} | {'Предсказанные 1':<15}")
print("-" * 75)

for threshold in thresholds:
    preds_custom = (y_proba >= threshold).astype(int)
    
    precision = precision_score(y_test, preds_custom, zero_division=0)
    recall = recall_score(y_test, preds_custom)
    f1 = f1_score(y_test, preds_custom)
    f2 = fbeta_score(y_test, preds_custom, beta=2)
    
    results.append({
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'f2': f2,
        'predicted_positives': preds_custom.sum()
    })
    
    print(f"{threshold:.2f}      | {precision:.4f}    | {recall:.4f}  | {f1:.4f}  | {f2:.4f}  | {preds_custom.sum():<15}")


In [ ]:
# Применение оптимального порога классификации для получения предсказаний и вычисление метрик качества модели
OPTIMAL_THRESHOLD = 0.39

new_preds = (y_proba >= OPTIMAL_THRESHOLD).astype(int)

print(f"Метрики модели с порогом {OPTIMAL_THRESHOLD}:")
print(f"Precision: {precision_score(y_test, new_preds):.2f}")
print(f"Recall: {recall_score(y_test, new_preds):.2f}")
print(f"F1-score = {f1_score(y_test, new_preds):.2f}")
print(f"F2-score = {fbeta_score(y_test, new_preds, beta=2):.2f}")
print(f"ROC-AUC = {roc_auc_score(y_test, y_proba):.2f}")


In [ ]:
# Визуализация матрицы ошибок (confusion matrix) для модели с оптимальным порогом классификации
plot_confusion_matrix(y_test, new_preds)


**Выводы:**

**Главная задача модели — выявить пользователей с высокой вероятностью покупки в течение 90 дней, чтобы повысить эффективность маркетинговых рассылок.**

 Основной приоритет — обеспечить максимально широкий охват потенциальных покупателей (максимизировать полноту — Recall), даже если это приведёт к значительному количеству ложных срабатываний (FP). Такой подход позволяет свести к минимуму упущенные возможности продаж.

**Оптимальной метрикой для оценки модели признан ROC-AUC,**
которая отражает способность модели правильно ранжировать пользователей по вероятности покупки. Значение 0.83 говорит о качественном отделении целевой аудитории от остальных, значительно превосходя случайный прогноз (ROC-AUC = 0.51 у DummyClassifier). Выбор порога классификации позволяет гибко регулировать баланс между точностью и полнотой в зависимости от бизнес-задач.

**Лучшая модель по результатам эксперимента — XGBClassifier**

с параметрами: learning_rate=0.1, max_depth=3, n_estimators=200, subsample=0.8, показавшая стабильный и высокий результат (средний ROC-AUC кросс-валидации — 0.677, ранг 1 среди всех моделей).

**Результаты на тестовой выборке с учётом оптимизации порога:**

Для достижения цели максимального охвата был выбран порог 0.39, при котором Recall достиг 0.90 — то есть модель выявляет 90% пользователей, которые действительно сделают покупку в течение периода, обеспечивая охват почти всей потенциальной аудитории.

Одновременно при этом Precision составляет 0.04 — это отражает компромисс, означающий, что среди пользователей, отобранных моделью, лишь 4% действительно совершают покупку. Исторгованных рассылок на «ложных» пользователей (43%) остаётся много, но это согласовано с задачей максимизировать захват потенциальных покупателей.

## Анализ важности признаков модели

### Проверка permutation importance

In [ ]:
# Вычисление важности признаков методом перестановочной важности (permutation importance) на тестовой выборке с метрикой ROC-AUC
permutation = permutation_importance(
    model, 
    X_test, 
    y_test, 
    scoring = 'roc_auc',
    n_repeats=10,
    random_state=RANDOM_STATE
)


In [ ]:
# Создание DataFrame с признаками и их средними значениями важности по результатам перестановочного анализа, отсортированными по убыванию важности
feature_importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance': permutation['importances_mean']
}).sort_values('importance', ascending=False)


In [ ]:
# Визуализация важности признаков на основе перестановочного анализа с описаниями полей
plot_permutation_importance(feature_importance, fields_description)


 - **Наиболее значимые признаки, оказывающие сильное влияние на вероятность совершения покупки в течение 90 дней:**

   - Идентификатор наиболее часто покупаемого клиентом товара (most_frequent_plu) с важностью 0.086

   - Количество взаимодействий с клиентом через мобильные push-уведомления (channel_mobile_push) — 0.083

   - Общее количество покупок наиболее часто приобретаемого товара (most_frequent_plu_count) — 0.065

   - Средний чек или средняя сумма покупки (avg_purchase_sum) — 0.064

   - Количество уникальных маркетинговых кампаний, вовлекавших клиента (unique_campaigns) — 0.042

   - Продолжительность клиентской истории в днях — разница между первой и последней покупкой (purchase_period_days) — 0.032

   - Частота контактов с клиентом через email (channel_email) — 0.022

   - Доля покупок самого частого товара относительно общего числа покупок клиента (most_frequent_plu_ratio) — 0.021

 - **Менее значимые признаки для модели:**

   - Количество уникальных товаров из ТОП-20 популярных позиций, приобретённых клиентом (unique_top20_plu) — 0.005

   - Количество уникальных товарных позиций (PLU), купленных клиентом (unique_plu_count) — 0.002

   - Количество уникальных дней покупок клиента (unique_purchase_days) — незначимое влияние с показателем -0.001

### Анализ SHAP (SHapley Additive exPlanations)

In [ ]:
# Размер выборки для анализа или эксперимента, установленный равным 200 объектам
SAMPLE_SIZE = 200


In [ ]:
# Обёртка для функции predict_proba, обеспечивающая конвертацию входных данных в DataFrame с нужными колонками перед предсказанием вероятностей
def predict_proba_wrapper(X):
    if not isinstance(X, pd.DataFrame):
        X = pd.DataFrame(X, columns=X_train.columns)
    return model.predict_proba(X)


In [ ]:
# Выбор случайной подвыборки из обучающих данных размером SAMPLE_SIZE для последующего анализа SHAP с фиксированным random_state для воспроизводимости
sample = shap.utils.sample(X_train, SAMPLE_SIZE, random_state=RANDOM_STATE)


In [ ]:
# Создание объяснителя SHAP типа KernelExplainer для интерпретации вероятностей модели с использованием логит-ссылки и выбранной подвыборки
explainer = shap.KernelExplainer(
    predict_proba_wrapper,
    sample,
    link="logit"
)


In [ ]:
# Выбор случайной подвыборки из тестовых данных размером SAMPLE_SIZE для анализа модели с фиксированным random_state для воспроизводимости
test_sample = X_test.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)


In [ ]:
# Получение SHAP-значений для выбранной тестовой подвыборки, объясняющих вклад признаков в предсказания модели
explanation = explainer(test_sample)


In [ ]:
# Извлечение SHAP-значений для положительного класса (индекс 1) из объекта объяснения модели
shap_values = explanation[:, :, 1]


#### График общей значимости признаков

In [ ]:
# Визуализация важности признаков на основе SHAP-значений с использованием описания полей
plot_shap_bar(shap_values, fields_description)


 - **Наиболее влиятельные признаки, определяющие вероятность совершения покупки в течение 90 дней:**

    - Общее число покупок наиболее часто приобретаемого товара у клиента (most_frequent_plu_count) с весом 0.32

    - Частота взаимодействий с пользователем через мобильные push-уведомления (channel_mobile_push) — 0.31

    - Средний размер одной покупки или средний чек (avg_purchase_sum) — 0.28

    - Идентификатор самого часто покупаемого товарного артикула (most_frequent_plu) — 0.21

    - Продолжительность покупательской активности клиента в днях (purchase_period_days) — 0.15

    - Доля покупок топового товара относительно всех покупок клиента (most_frequent_plu_ratio) — 0.13

    - Количество уникальных маркетинговых кампаний, в которых участвовал клиент (unique_campaigns) — 0.13

    - Частота попадания клиента в email-кампании (channel_email) — 0.10

 - **Менее значимые для модели признаки:**

    - Количество уникальных товаров из ТОП-20 популярных, приобретённых клиентом (unique_top20_plu) — 0.03

    - Число уникальных товарных позиций (PLU), купленных клиентом (unique_plu_count) — 0.01

    - Число уникальных дней покупок (unique_purchase_days) — незначимое отрицательное влияние (-0.01)

#### График влияния признаков на предсказания модели

In [ ]:
# Построение графика "пчелиный улей" (beeswarm) для SHAP-значений, отображающего распределение влияния признаков на предсказания модели с описаниями полей
plot_shap_beeswarm(shap_values, test_sample, fields_description)


**Итоги анализа с использованием Permutation Importance и SHAP (SHapley Additive exPlanations)**

**Основные признаки, оказывающие наибольшее влияние на вероятность покупки клиента в целевом периоде:**

- Количество покупок самого часто приобретаемого товара (most_frequent_plu_count): Чем больше покупок этого товара, тем выше вероятность совершения покупки клиентом.

- Частота взаимодействия с клиентом посредством push-уведомлений на мобильное устройство (channel_mobile_push): Увеличение таких взаимодействий повышает вероятность покупки.

- Средний чек (avg_purchase_sum): Меньшая средняя стоимость покупки ассоциируется с большей вероятностью совершения покупки.

- Идентификатор самого часто покупаемого товара (most_frequent_plu): Существует сложная взаимосвязь между этим признаком и вероятностью покупки.

- Длительность клиентской истории (purchase_period_days): Чем дольше период между первой и последней покупкой, тем выше шансы на повторную покупку.

- Доля покупок самого частого товара от общего числа покупок (most_frequent_plu_ratio): Меньшая доля этого товара связана с увеличением вероятности покупки.

- Количество уникальных маркетинговых кампаний, в которых участвовал клиент (unique_campaigns): Большее число кампаний повышает вероятность покупки.

- Частота взаимодействия с клиентом через email (channel_email): Влияет неоднозначно — может как повышать, так и снижать вероятность покупки.

**Признаки с низкой значимостью для модели:**

- Количество уникальных популярных товаров из ТОП-20, приобретённых клиентом (unique_top20_plu).

- Общее число уникальных товарных позиций, купленных клиентом (unique_plu_count).

- Количество уникальных дней, в которые клиент совершал покупки (unique_purchase_days).

## Выводы

### Описание проведенной предобработки данных 

**Выводы:**

- **Проверка типов данных**:
    - Данные о покупках клиентов по дням и по товарам.
        - В поле price преобразован тип данных из float64 в int64, так как данный показатель является ценой, которая не имеет дробной части.

- **Проверка пропусков**:
    - Данные о покупках клиентов по дням и по товарам.
        - В данных отсутствуют пропуски.
    - Данные о рассылках, которые были отправлены клиентам из таблицы покупок.
        - В данных отсутствуют пропуски.

- **Проверка дубликатов**:
    - **Полные дубликаты:**
        - Данные о покупках клиентов по дням и по товарам.
            - Выявлено 73.020 / 202.208 (36%) полных дубликтов в данных о покупках клиентов по дням и по товарам.
            Однако, данные дубликаты могут быть не техническими, а семантическими, то есть внешне похожие записи, которые могут отражать:
                - разные характеристи товара в одном заказе: цвет, размер, модификация;
                - разные товары одной категории;
                - повторные покупки идентичных товаров в течение дня.
        - Данные о рассылках, которые были отправлены клиентам из таблицы покупок.
            - Выявлено 48.610 / 12.739.798 (0.4%) полных дубликтов в данных о рассылках, которые были отправлены клиентам из таблицы покупок.
            Для данных дубликатов отсутствуют достаточные логические объянения их наличия, поэтому они удалены из совокупности.

        
    - **Частичные дубликаты:**
        - Данные о покупках клиентов по дням и по товарам.
            - Во всех столбцах допустимо наличие значений дубликатов.
        - Данные о рассылках, которые были отправлены клиентам из таблицы покупок.
            - Во всех столбцах допустимо наличие значений дубликатов.

### Исследовательский анализ данных (до агрегации)

**Выводы:**

- **Данные о рассылках, которые были отправлены клиентам из таблицы покупок:**

    - Признак **'message_id'** является хэш-значением. В данных присутствуют записи с одинаковыми значениями 'message_id'.

    - Признак **'client_id'** является целочисленным идентификатором клиента. В данных есть записи с одинаковыми значениями 'client_id'.

    - Признак **'bulk_campaign_id'** является целочисленным идентификатором рассылки. В рамках одной рассылки отправляется большое количество сообщений.

    - Признак **'event'** показывает, какое действие было совершено с сообщением. ТОП-3 действия: 'send' (71,4%), 'open' (24,1%) и 'click' (3,8%) составляют 99% событий.

    - Признак **'channel'** показывает, через какой канал была отправлена рассылка: 'mobile push' (58,9%) или 'email' (41,1%).

    - Признаки **'message_date'** и **'message_created_at'** показывают дату отправки сообщения, при этом 'message_created_at' включает также и время отправки. При анализе дат отправки сообщений в разрезе дней и недель наблюдается тренд на значительное увеличение среднего ежедневного и еженедельного количества отправленных сообщений, начиная со второго полугодия 2023 года.

- **Данные о покупках клиентов по дням и по товарам:**

    - Признак **'client_id'** является целочисленным идентификатором клиента. В данных присутствуют записи с одинаковыми 'client_id'.

    - Признак **'category_ids'** представляет собой список идентификаторов категорий, к которым относится товар. Он может быть пустым. А также поскольку дерево категорий обновляется, могут меняться и их вложенности. Для анализа и создания новых признаков будет использоваться последний (листовой) уровень категории, который, скорее всего, является кодом товара (PLU) или максимально близкой к нему категорией.

    - Признак **'quantity'** показывает количество купленного товара. Среднее значение и медиана данного признака равны 1, что означает: в большинстве случаев покупатели приобретают по 1 единице каждого товара.

    - Признак **'price'** отражает цену купленного товара. Среднее значение равно 1199 ед., что превышает медианное значение - 987 ед. Это говорит о наличии выбросов — покупок товаров с ценой, значительно превышающей медианную.

    - Признак **'purchase_date'** указывает дату совершения покупки. Анализ данных в разрезе дней и недель показывает тренд на значительное снижение среднего ежедневного и еженедельного количества покупок, начиная со второго полугодия 2023 года.

### Исследовательский анализ данных (после агрегации)

**Выводы:**

- Признак **'client_id'** - идентификатор клиента. Является первичным ключом в данных, подготовленных для обучения ML-модели.

- Признак **'most_frequent_plu'** - идентификатор товарной позиции (PLU), которую клиент покупал чаще всего. ТОП-20 товаров охватывает  44% от общего количества проданных товаров. 

- Признак **'unique_campaigns'** - количество уникальных маркетинговых кампаний, в которые был включен клиент. Среднее количество маркетинговых компаний, в которые был включен клиент - 156.89, а медианное - 150, что говорит о наличии выбросов: клиентов, которым были разосланы сообщения в рамках значительно большего количества маркетинговых компаний.

- Признак **'channel_email'** - количество взаимодействий с клиентом через электронную почту и признак **'channel_mobile_push'** - количество взаимодействий с клиентом через push-уведомления на мобильное устройство. Медианное значение количества взаимодействий через электронную почту и уведомления на мобильное устройство примерно равное: 129 (для 'электронная почта') и 125 (для 'push-уведомления на мобильное устройство'). Для обоих каналов среднее значение превышает медианное, что говорит о наличии выбросов: клиентов, получивших значительно большее количество сообщений.

- Признак **'avg_purchase_sum'** - средняя стоимость одной покупки (средний чек). Средние значения 'среднего чека' - 16.012,медианное значение - 8.396. Значительная разница (примерно двукратная) между средним и медианным значением говорит о большом количестве выбросов: покупок, стоимость которых значительно превышает медианную.

- Признак **'unique_purchase_days'** - количество уникальных дней, в которые клиент совершал покупки и признак **'purchase_period_days'** - количество дней между первой и последней покупкой клиента (длина клиентской истории). Как видно из значений данных признаков, особенно из медианных: количество уникальных дней с покупками - 1 день, а длина истории - 0 дней, — можно заключить, что большинство покупателей не совершают повторных покупок.

- Признак **'unique_plu_count'** - количество уникальных товарных позиций (PLU), которые покупал клиент. Как видно из медианного значения данного признака - 1, большинство покупателей покупает товар, который относится только к одной уникальной товарной позиции. А среднее значение - 1.83, которое практически в два раза превышает медиану говорит о наличии выбросов, покупателей которые покупают значительно больше разных уникальных товарных позиций.
- Признак **'unique_top20_plu'** - количество уникальных товаров из общего ТОП-20 популярных товаров, которые покупал клиент. Медианное значение количества уникальных товаров из общего ТОП-20 популярных товаров, которые покупал клиент равно 1, а среднее - 0.71, что говорит о том, что покупатели покупают в основном товары, относящиеся только к одной товарной позиции.

- Признак **'most_frequent_plu_count'** - количество покупок самого частого товара у клиента. Медианное количество покупок самого частого товара у клиента - 6, а среднее 9.84, что снова указывает на наличие выбросов.
- Признак **'most_frequent_plu_ratio'** - доля покупок самого частого товара от общего числа покупок клиента. Медианное значение доли покупок самого частого товара от общего числа покупок клиента - 100%, а среднее - 80%. Это подтверждает вывод, что для большинства клиентов все покупки или их большая часть приходятся на одну товарную позицию.

### Корреляционный анализ признаков

Отсутствие сильных линейных корреляций с целевой переменной свидетельствует о сложной, нелинейной природе взаимосвязей между признаками и фактом совершения покупки в целевом периоде. 

Наблюдаются выраженные взаимосвязи между созданными на основе исходных данных признаками, что указывает на комплексный характер признаков.

### Поиск лучшей модели машинного обучения

**Основные шаги поиска лучшей модели машинного обучения:**

* инициализация и подготовка данных

* создание пайплайнов предобработки данных и обучения

* определение гиперпараметров для обучения моделей

* выбор лучшей модели машинного обучения

* анализ результатов и подбор оптимального порога классификации

* оценка модели машинного обучения на тестовой выборке

### Выбранная модель машинного обучения

**Цель модели — выявить пользователей с высокой вероятностью совершить покупку в течение 90 дней для оптимизации маркетинговых коммуникаций:**

- Ключевой приоритет — максимально полный охват (Recall) потенциальных покупателей, чтобы маркетинговые предложения достигли большинства из них. Допустимо включение в целевую аудиторию части пользователей, которые не совершат покупку (FP - ложные срабатывания), чтобы минимизировать риск упустить реальные продажи.

**Для оценки модели лучше всего подходит метрика ROC-AUC:**

ROC-AUC демонстрирует общую способность модели ранжировать пользователей по вероятности совершения покупки. Значение 0.83 указывает на хорошее качество разделения, значительно превосходящее случайный алгоритм (DummyClassifier, ROC-AUC=0.51). Итоговый баланс между точностью предсказаний и полнотой охвата определяется бизнес-требованиями и оптимизируется выбором порога классификации.

**Лучшая модель:** XGBClassifier

**Гиперпараметры и кросс-валидация:**

Алгоритм: Градиентный бустинг (XGBoost).

Ключевые параметры: learning_rate=0.1, max_depth=3, n_estimators=200, subsample=0.8.

Средний результат кросс-валидации (ROC-AUC): 0.677.

Выбранная модель показала наилучшую и стабильную производительность среди протестированных (rank_test_score=1).

**Метрики на тестовой выборке и анализ после подбора порога:**

Для соответствия бизнес-цели (максимально не упустить покупателей) был подобран оптимальный порог 0.39, который сместил баланс метрик:

Recall вырос до 0.90: Теперь модель идентифицирует 90% всех пользователей, которые совершат покупку в целевой период. Это позволяет маркетинговой кампании охватить практически всех потенциальных покупателей.

Precision остался низким (0.04): Это отражает компромисс — лишь 4% пользователей в целевой выборке действительно совершат покупку. Рассылка будет направлена большому числу пользователей (FP-43%), которые не конвертируются в ближайшие 90 дней.

### Анализ важности признаков модели

**Итоги анализа с использованием Permutation Importance и SHAP (SHapley Additive exPlanations)**

**Основные признаки, оказывающие наибольшее влияние на вероятность покупки клиента в целевом периоде:**

- Количество покупок самого часто приобретаемого товара (most_frequent_plu_count): Чем больше покупок этого товара, тем выше вероятность совершения покупки клиентом.

- Частота взаимодействия с клиентом посредством push-уведомлений на мобильное устройство (channel_mobile_push): Увеличение таких взаимодействий повышает вероятность покупки.

- Средний чек (avg_purchase_sum): Меньшая средняя стоимость покупки ассоциируется с большей вероятностью совершения покупки.

- Идентификатор самого часто покупаемого товара (most_frequent_plu): Существует сложная взаимосвязь между этим признаком и вероятностью покупки.

- Длительность клиентской истории (purchase_period_days): Чем дольше период между первой и последней покупкой, тем выше шансы на повторную покупку.

- Доля покупок самого частого товара от общего числа покупок (most_frequent_plu_ratio): Меньшая доля этого товара связана с увеличением вероятности покупки.

- Количество уникальных маркетинговых кампаний, в которых участвовал клиент (unique_campaigns): Большее число кампаний повышает вероятность покупки.

- Частота взаимодействия с клиентом через email (channel_email): Влияет неоднозначно — может как повышать, так и снижать вероятность покупки.

**Признаки с низкой значимостью для модели:**

- Количество уникальных популярных товаров из ТОП-20, приобретённых клиентом (unique_top20_plu).

- Общее число уникальных товарных позиций, купленных клиентом (unique_plu_count).

- Количество уникальных дней, в которые клиент совершал покупки (unique_purchase_days).